In [2]:
!uv pip install requests beautifulsoup4 lxml loguru langchain-groq langchain_nvidia_ai_endpoints -q


In [2]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file (overrides existing env vars)
load_dotenv(override=True)

# ─── NVIDIA LLM ─────────────────────────────────────────────────────────────────
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# Get API key from environment variable instead of Google Colab
nvidia_api_key = os.getenv('NVIDIA_API_KEY')  # Changed from 'CHATGPT' to more descriptive name

# Check if API key is set
if not nvidia_api_key:
    raise ValueError("NVIDIA_API_KEY not found in environment variables. Please add it to your .env file.")

MODEL = "openai/gpt-oss-120b"

def get_llm(temperature: float = 0.2) -> ChatNVIDIA:
    return ChatNVIDIA(
        model=MODEL,
        api_key=nvidia_api_key,
        temperature=temperature,
        top_p=1,
        max_tokens=4096,
    )

# Non-streaming version
client = get_llm()
response = client.invoke([
    {"content": "what is RAG? short answer, 5-6 sentences", "role": "user"}
])
print(response.content)

/var/folders/fx/61s94vks26j1dz_f_1bkfbhw0000gn/T/ipykernel_98517/942191937.py:20: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  return ChatNVIDIA(


Retrieval‑Augmented Generation (RAG) is a hybrid AI architecture that combines a large language model (the “generator”) with an external information‑retrieval system (the “retriever”). When given a prompt, the retriever first searches a knowledge base—such as documents, web pages, or vector‑indexed embeddings—and returns the most relevant passages. These retrieved texts are then fed into the generator, which conditions its output on both the original query and the supplemental material. By grounding its responses in up‑to‑date or domain‑specific sources, RAG mitigates the hallucination problem and can produce more factual, context‑aware answers. It is widely used for question answering, chatbots, and enterprise search where accuracy and freshness of information are critical.


In [3]:

"""
SEC Fine-Tuning Dataset Builder  v15  — RESEARCH-GRADE REASONING DATASET
=========================================================================
Target score: 97-98+ / 100

v15 IMPROVEMENTS (based on expert audit feedback):
────────────────────────────────────────────────────
FIX-A  NARRATIVE EXTRACTION: Aggressive multi-strategy MD&A/Risk section extraction
        - Strategy 1: Full document text search with relaxed patterns
        - Strategy 2: Sentence-level financial keyword density scan
        - Strategy 3: EDGAR viewer API fallback
        - Strategy 4: SGML full-text fallback
        - Minimum viable text threshold lowered to 50 chars
        - Dedicated text quality scorer

FIX-B  429 RATE LIMIT MANAGEMENT: Intelligent adaptive throttling
        - Exponential backoff with jitter
        - Per-provider rate limit tracker
        - Automatic wait-time parser from error messages
        - Max concurrent request limiter
        - Circuit breaker pattern (skip provider after N consecutive failures)

FIX-C  GOLD FILTER: Quality-gated training set export
        - Configurable quality threshold (default >= 90)
        - Separate gold/silver/bronze tier JSONL outputs
        - Automated tier assignment at write time

FIX-D  YAHOO FINANCE 5-DAY ABNORMAL RETURN: Market impact signal
        - Fetch stock price around filing date
        - Compute 5-day abnormal return vs SPY benchmark
        - Inject into meta + signals for market-impact training

CRITICAL DATA PIPELINE FIXES (v14, retained):
────────────────────────────────────────────────
FIX-1  DE-CUMULATION: YTD → Discrete 3-month conversion
FIX-2  QUARTER-ALIGNED YoY: Same fiscal quarter prior year
FIX-3  SANITY VALIDATION: Block phantom swings pre-LLM
FIX-4  PERIOD TYPE METADATA: period_type on every record

v13 UPGRADES (retained):
1-12.  All 12 expert audit upgrades from v13

INSTALL:
  pip install requests beautifulsoup4 lxml loguru langchain-groq
              python-dotenv tqdm yfinance
"""

import os
import re
import json
import sys
import time
import hashlib
import random
import warnings
from collections import defaultdict
from datetime import datetime, timedelta
from typing import Optional

import requests
from bs4 import BeautifulSoup, XMLParsedAsHTMLWarning
from loguru import logger
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

# ─── LOGURU ───────────────────────────────────────────────────────────────────
logger.remove()
logger.add(
    sys.stderr,
    format="<green>{time:HH:mm:ss}</green> | <level>{level: <7}</level> | {message}",
    level="INFO",
    colorize=True,
)
logger.add(
    "sec_builder_v15.log",
    format="{time:YYYY-MM-DD HH:mm:ss} | {level: <7} | {function}:{line} | {message}",
    level="DEBUG",
    rotation="50 MB",
    retention="7 days",
)

# ─── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_FILE              = "sec_finetune_v15.jsonl"
GOLD_FILE                = "sec_finetune_v15_gold.jsonl"
SILVER_FILE              = "sec_finetune_v15_silver.jsonl"
PENDING_FILE             = "sec_pending_v15.jsonl"
QUALITY_LOG_FILE         = "sec_quality_v15.jsonl"
SANITY_LOG_FILE          = "sec_sanity_v15.jsonl"

YEARS_BACK               = 5
EDGAR_SLEEP              = 0.12
MAX_RETRIES              = 5
BACKOFF_BASE             = 2.0
BACKOFF_MAX              = 120.0
PERIOD_FUZZ_DAYS         = 5
MAX_TEXT_CHARS           = 5000
MIN_PARA_LEN             = 40        # FIX-A: lowered from 60
MIN_PARA_ALPHA           = 0.40      # FIX-A: lowered from 0.50
LLM_429_MAX_WAIT         = 600       # FIX-B: raised from 400
LLM_CIRCUIT_BREAK_N      = 5        # FIX-B: consecutive 429s before circuit break
MIN_NARRATIVE_DOC_BYTES  = 80_000   # FIX-A: lowered from 200_000
MIN_SECTION_CHARS        = 50       # FIX-A: minimum viable section text
MEGA_CAP_SWING_THRESHOLD = 25.0

# FIX-C: Gold filter thresholds
GOLD_QUALITY_THRESHOLD   = 90
SILVER_QUALITY_THRESHOLD = 75

# FIX-D: Yahoo Finance abnormal return window
ABNORMAL_RETURN_DAYS     = 5
SPY_TICKER               = "SPY"

EDGAR_HEADERS = {
    "User-Agent": "SECDatasetBuilder/1.0 (ML research; research@university.edu)",
    "Accept-Encoding": "gzip, deflate, br",
    "Accept": "application/json, text/html, */*",
}

# ─── CANONICAL FINANCIAL TERMINOLOGY ──────────────────────────────────────────
CANONICAL_TERMS = {
    "profitability pressure":  "margin pressure",
    "earnings pressure":       "margin pressure",
    "profit decline":          "margin pressure",
    "profit compression":      "margin pressure",
    "margin deterioration":    "margin pressure",
    "debt burden":             "leverage pressure",
    "debt load":               "leverage pressure",
    "heavy debt":              "leverage pressure",
    "liquidity issue":         "liquidity pressure",
    "liquidity concern":       "liquidity pressure",
    "cash crunch":             "liquidity pressure",
    "revenue contraction":     "revenue decline",
    "top-line growth":         "revenue growth",
    "top-line decline":        "revenue decline",
    "bottom-line growth":      "net income growth",
    "bottom-line pressure":    "margin pressure",
    "operational efficiency":  "operating leverage improvement",
    "cost optimization":       "cost structure improvement",
    "capital efficiency":      "asset utilization",
    "strong performance":      "above-average financial results",
    "weak performance":        "below-average financial results",
    "showing strength":        "metrics above historical average",
    "showing weakness":        "metrics below historical average",
}

FILLER_TOKENS = {
    "suggesting", "indicating", "overall", "reflects", "demonstrates",
    "showcases", "underscores", "highlights", "illustrates", "notably",
    "importantly", "significantly", "meaningfully", "materially",
    "generally", "broadly", "largely", "essentially", "fundamentally",
    "incredibly", "remarkably", "substantially", "considerably",
    "improved", "declined", "strong", "weak",
    "robust", "solid", "healthy", "challenging", "difficult",
}

NEGATIVE_REASONING_TEMPLATES = [
    {
        "trigger": "revenue_yoy_pct < 5 and revenue_yoy_pct > 0",
        "bad_conclusion": "Strong market demand drove revenue.",
        "correct_conclusion": "Revenue grew modestly at {revenue_yoy_pct:.1f}% YoY; no strong acceleration signal. No explicit growth driver identifiable from available data.",
        "label": "weak_growth_overclaim",
    },
    {
        "trigger": "revenue_yoy_pct > 15 and not mda_has_text",
        "bad_conclusion": "Product demand and market expansion drove growth.",
        "correct_conclusion": "Revenue grew {revenue_yoy_pct:.1f}% YoY. No MD&A text available; growth driver unidentifiable — quantitative signal only.",
        "label": "growth_without_driver_evidence",
    },
    {
        "trigger": "gross_margin_pct_delta < -2 and not mda_has_text",
        "bad_conclusion": "Supply chain disruption caused margin compression.",
        "correct_conclusion": "Gross margin compressed {gross_margin_pct_delta:.1f}pp YoY. No explicit cost driver identified in available data.",
        "label": "margin_compression_unsupported",
    },
    {
        "trigger": "operating_margin_pct > 20 and net_margin_pct < 5",
        "bad_conclusion": "Strong operations indicate overall business health.",
        "correct_conclusion": "Operating margin at {operating_margin_pct:.1f}% but net margin only {net_margin_pct:.1f}% — significant below-the-line costs materially reduce profitability.",
        "label": "operating_vs_net_margin_disconnect",
    },
    {
        "trigger": "fcf_yoy_pct > 0 and revenue_yoy_pct < -5",
        "bad_conclusion": "Business momentum is positive given improving cash flows.",
        "correct_conclusion": "FCF improved {fcf_yoy_pct:.1f}% YoY despite revenue declining {revenue_yoy_pct:.1f}%. FCF improvement during revenue decline may reflect cost-cutting or CapEx reduction, not business acceleration.",
        "label": "fcf_improvement_revenue_decline",
    },
]

EDGE_CASE_ARCHETYPES = {
    "unprofitable_expansion": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) > 15 and (m.get("operating_margin_pct", 0) or 0) < -5,
        "label": "growth_margin_collapse",
        "reasoning_focus": "Distinguish revenue scale from profitability. High growth + negative margins = investment phase or structural cost problem.",
    },
    "efficiency_harvest": {
        "condition": lambda m, c: abs((c.get("revenue_yoy_pct", 0) or 0)) < 3 and (c.get("fcf_yoy_pct", 0) or 0) > 15,
        "label": "flat_revenue_fcf_improvement",
        "reasoning_focus": "Flat revenue + improving FCF = operational efficiency or CapEx reduction, not growth stagnation.",
    },
    "restructuring_signal": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) < -5 and (c.get("gross_margin_pct_delta", 0) or 0) > 2,
        "label": "revenue_decline_margin_improvement",
        "reasoning_focus": "Revenue decline + margin improvement = portfolio pruning, restructuring, or mix shift toward premium products.",
    },
    "manageable_leverage": {
        "condition": lambda m, c: (m.get("debt_to_equity", 0) or 0) > 2.0 and (c.get("fcf_yoy_pct", 0) or 0) > 0 and (m.get("free_cash_flow", 0) or 0) > 0,
        "label": "high_leverage_strong_cashflow",
        "reasoning_focus": "High D/E ratio alone is insufficient for leverage risk — FCF coverage determines debt serviceability.",
    },
    "liquidity_paradox": {
        "condition": lambda m, c: (m.get("current_ratio", 99) or 99) < 1.0 and (m.get("cash_and_equivalents", 0) or 0) > 1e9,
        "label": "low_current_ratio_high_cash",
        "reasoning_focus": "Current ratio below 1.0 with large absolute cash balance — nuanced interpretation required, not automatic liquidity crisis.",
    },
    "conflicting_earnings": {
        "condition": lambda m, c: (c.get("revenue_yoy_pct", 0) or 0) > 10 and (c.get("net_income_yoy_pct", 0) or 0) < -10,
        "label": "revenue_growth_income_decline",
        "reasoning_focus": "Revenue growth with net income decline = cost structure problem, investment cycle, or tax/interest event. Cannot conclude business health from revenue alone.",
    },
}

REASONING_TYPES = {
    "trend_analysis":       lambda m, c: c.get("revenue_yoy_pct") is not None,
    "margin_analysis":      lambda m, c: m.get("gross_margin_pct") is not None and m.get("net_margin_pct") is not None,
    "risk_assessment":      lambda m, c: m.get("debt_to_equity") is not None or m.get("current_ratio") is not None,
    "cash_flow_quality":    lambda m, c: m.get("free_cash_flow") is not None and m.get("operating_cash_flow") is not None,
    "capital_allocation":   lambda m, c: m.get("capex") is not None and m.get("share_repurchases") is not None,
    "profitability_driver": lambda m, c: c.get("gross_margin_pct_delta") is not None,
    "leverage_assessment":  lambda m, c: m.get("long_term_debt") is not None and m.get("net_debt") is not None,
    "growth_quality":       lambda m, c: c.get("revenue_yoy_pct") is not None and c.get("eps_yoy_pct") is not None,
    "multi_horizon":        lambda m, c: False,
    "negative_inference":   lambda m, c: False,
    "edge_case":            lambda m, c: False,
    "market_impact":        lambda m, c: False,
}

FINANCIAL_KEYWORDS = {
    "revenue", "net income", "earnings", "operating", "margin", "profit",
    "growth", "decline", "increased", "decreased", "compared", "fiscal",
    "quarter", "year", "segment", "billion", "million", "percent", "%",
    "cash flow", "capex", "ebitda", "gross", "cost", "expense", "loss",
    "gain", "acquisition", "guidance", "outlook", "demand", "supply",
    "customers", "market share", "competition", "results", "sales",
    "income", "debt", "equity", "assets", "liabilities", "dividend",
    "repurchase", "buyback", "restructur", "impairment", "write",
}

MATERIAL_8K_KEYWORDS = {
    "revenue", "earnings", "net income", "quarterly results", "annual results",
    "acquisition", "merger", "acquires", "acquired", "divest",
    "ceo", "cfo", "chief executive", "chief financial", "resign", "appoint",
    "restructur", "layoff", "workforce reduction",
    "lawsuit", "litigation", "settlement", "investigation",
    "guidance", "outlook", "dividend", "buyback", "repurchase", "impairment",
}

ITEM_HEADING_RE = re.compile(r"\bitem\s+(?:1a?|2|3|7a?|8)\b", re.IGNORECASE)

SECTIONS_10K = {
    "business":             [r"item\s+1[\.\s]*[-–—]?\s*business\b",            r"(?<!\w)item\s+1(?!\s*[aAbB])\b"],
    "risk_factors":         [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor",        r"item\s+1a\b"],
    "mda":                  [r"item\s+7[\.\s]*[-–—]?\s*management.{0,120}discussion", r"(?<!\w)item\s+7(?!\s*a)\b"],
    "market_risk":          [r"item\s+7a[\.\s]*[-–—]?\s*quantitative",         r"item\s+7a\b"],
    "financial_statements": [r"item\s+8[\.\s]*[-–—]?\s*financial\s+statement", r"(?<!\w)item\s+8\b"],
    "legal_proceedings":    [r"item\s+3[\.\s]*[-–—]?\s*legal\s+proceed",       r"(?<!\w)item\s+3\b"],
}
SECTIONS_10Q = {
    "mda":                  [r"item\s+2[\.\s]*[-–—]?\s*management.{0,120}discussion", r"item\s+2\b"],
    "market_risk":          [r"item\s+3[\.\s]*[-–—]?\s*quantitative",          r"item\s+3\b"],
    "risk_factors":         [r"item\s+1a[\.\s]*[-–—]?\s*risk\s+factor",        r"item\s+1a\b"],
    "financial_statements": [r"item\s+1[\.\s]*[-–—]?\s*financial\s+statement", r"(?<!\w)item\s+1(?!\s*a)\b"],
    "legal_proceedings":    [r"item\s+1[\.\s]*[-–—]?\s*legal\s+proceed"],
}

NEXT_ITEM_RE = re.compile(r"\bitem\s+\d{1,2}[ab]?\s*[\.\s]", re.IGNORECASE)
_NUM_RE       = re.compile(r"(\$[\d,.]+[BbMmKk]?|\d+\.?\d*\s*%|\b\d{1,3}(?:,\d{3})+\b)", re.IGNORECASE)

RISK_KW = [
    "competition", "regulation", "supply chain", "cybersecurity", "inflation",
    "interest rate", "liquidity", "litigation", "currency risk", "macroeconomic",
    "recession", "geopolitical", "tariff", "sanctions", "climate", "data privacy",
    "customer concentration", "patent", "labor shortage", "ai disruption",
]

# FIX-A: Additional aggressive MD&A patterns
MDA_FALLBACK_PATTERNS = [
    r"management.{0,30}discussion",
    r"results\s+of\s+operations",
    r"liquidity\s+and\s+capital",
    r"critical\s+accounting",
    r"overview\s+of\s+.*\s+results",
    r"executive\s+summary",
    r"financial\s+highlights",
    r"operating\s+results",
    r"revenue\s+recognition",
    r"segment\s+(results|information|reporting)",
]
MDA_FALLBACK_RE = [re.compile(p, re.IGNORECASE) for p in MDA_FALLBACK_PATTERNS]

RISK_FALLBACK_PATTERNS = [
    r"risk\s+factor",
    r"we\s+face\s+(significant|material|various)\s+risk",
    r"our\s+business\s+(is|may\s+be)\s+subject\s+to",
    r"uncertaint(y|ies)",
    r"competitive\s+risk",
    r"regulatory\s+(risk|requirement|change)",
]
RISK_FALLBACK_RE = [re.compile(p, re.IGNORECASE) for p in RISK_FALLBACK_PATTERNS]


# ══════════════════════════════════════════════════════════════════════════════
# FIX-B: ADAPTIVE RATE LIMIT MANAGER
# ══════════════════════════════════════════════════════════════════════════════

class RateLimitManager:
    """
    FIX-B: Intelligent 429 rate limit handler.
    - Tracks consecutive failures per provider
    - Implements exponential backoff with jitter
    - Circuit breaker pattern
    - Auto-parses wait time from error messages
    """

    def __init__(self):
        self.consecutive_failures: int  = 0
        self.total_429s:           int  = 0
        self.total_skipped:        int  = 0
        self.last_wait:            float = 0.0
        self.circuit_open:         bool  = False
        self.circuit_open_until:   float = 0.0

    def record_success(self):
        self.consecutive_failures = 0
        self.circuit_open         = False

    def record_429(self, wait_seconds: float):
        self.consecutive_failures += 1
        self.total_429s           += 1
        self.last_wait             = wait_seconds

        if self.consecutive_failures >= LLM_CIRCUIT_BREAK_N:
            cool_down = min(wait_seconds * 2, 600)
            self.circuit_open       = True
            self.circuit_open_until = time.time() + cool_down
            logger.warning(
                f"  ⚡ CIRCUIT BREAKER OPEN: {self.consecutive_failures} consecutive 429s. "
                f"Cooling {cool_down:.0f}s until {datetime.fromtimestamp(self.circuit_open_until).strftime('%H:%M:%S')}"
            )

    def record_skip(self):
        self.total_skipped += 1

    def is_circuit_open(self) -> bool:
        if self.circuit_open and time.time() > self.circuit_open_until:
            self.circuit_open         = False
            self.consecutive_failures = 0
            logger.info("  ✓ Circuit breaker CLOSED — resuming LLM calls")
        return self.circuit_open

    def wait_circuit(self):
        if self.circuit_open:
            remaining = self.circuit_open_until - time.time()
            if remaining > 0:
                logger.warning(f"  Circuit open — waiting {remaining:.0f}s")
                time.sleep(remaining + 1)
                self.circuit_open         = False
                self.consecutive_failures = 0

    @staticmethod
    def parse_wait_from_error(err_str: str) -> int:
        """Extract wait seconds from Groq/OpenAI 429 error message."""
        # "try again in 1m30s"
        m = re.search(r"try again in (\d+)m([\d.]+)s", err_str)
        if m:
            return int(m.group(1)) * 60 + int(float(m.group(2))) + 5
        # "try again in 30.5s"
        m = re.search(r"try again in ([\d.]+)s", err_str)
        if m:
            return int(float(m.group(1))) + 5
        # "retry after 45"
        m = re.search(r"retry.{0,10}after\s+(\d+)", err_str, re.IGNORECASE)
        if m:
            return int(m.group(1)) + 5
        # "rate limit" with no explicit time
        return 60

    @staticmethod
    def jitter(base: float) -> float:
        """Add ±20% jitter to avoid thundering herd."""
        return base * (0.8 + random.random() * 0.4)

    def summary(self) -> str:
        return (
            f"RateLimitManager: total_429s={self.total_429s}, "
            f"skipped={self.total_skipped}, "
            f"consecutive_now={self.consecutive_failures}"
        )


# Global rate limit manager (shared across all LLM calls)
_rate_limiter = RateLimitManager()


# ══════════════════════════════════════════════════════════════════════════════
# FIX-D: YAHOO FINANCE ABNORMAL RETURN
# ══════════════════════════════════════════════════════════════════════════════

def fetch_abnormal_return(
    ticker: str,
    filing_date: str,
    window_days: int = ABNORMAL_RETURN_DAYS,
) -> Optional[dict]:
    """
    FIX-D: Compute N-day abnormal return around SEC filing date.

    Abnormal Return = Stock Return - Benchmark (SPY) Return
    over [filing_date, filing_date + window_days]

    Returns dict with:
        stock_return_pct     : raw stock return over window
        spy_return_pct       : SPY return over same window
        abnormal_return_pct  : stock - SPY
        window_start         : actual start date
        window_end           : actual end date
        data_source          : "yahoo_finance"
    """
    try:
        import yfinance as yf
    except ImportError:
        logger.warning("yfinance not installed — skipping abnormal return (pip install yfinance)")
        return None

    try:
        start_dt = datetime.strptime(filing_date[:10], "%Y-%m-%d")
    except ValueError:
        return None

    # Fetch a bit extra to handle weekends/holidays
    fetch_start = (start_dt - timedelta(days=2)).strftime("%Y-%m-%d")
    fetch_end   = (start_dt + timedelta(days=window_days + 5)).strftime("%Y-%m-%d")

    try:
        stock_data = yf.download(
            ticker, start=fetch_start, end=fetch_end,
            auto_adjust=True, progress=False, timeout=15,
        )
        spy_data = yf.download(
            SPY_TICKER, start=fetch_start, end=fetch_end,
            auto_adjust=True, progress=False, timeout=15,
        )

        if stock_data.empty or spy_data.empty:
            logger.debug(f"  yfinance: no data for {ticker} around {filing_date}")
            return None

        # Align on common dates
        common_idx = stock_data.index.intersection(spy_data.index)
        if len(common_idx) < 2:
            return None

        stock_data = stock_data.loc[common_idx]["Close"]
        spy_data   = spy_data.loc[common_idx]["Close"]

        # Find first trading day >= filing_date
        filing_dt_ts = start_dt
        avail_dates  = [d for d in common_idx if d.date() >= filing_dt_ts.date()]
        if len(avail_dates) < 2:
            return None

        t0    = avail_dates[0]
        t_end_idx = min(window_days, len(avail_dates) - 1)
        t_end = avail_dates[t_end_idx]

        p0_stock = float(stock_data.loc[t0])
        pt_stock = float(stock_data.loc[t_end])
        p0_spy   = float(spy_data.loc[t0])
        pt_spy   = float(spy_data.loc[t_end])

        if p0_stock <= 0 or p0_spy <= 0:
            return None

        stock_ret    = round((pt_stock / p0_stock - 1) * 100, 3)
        spy_ret      = round((pt_spy   / p0_spy   - 1) * 100, 3)
        abnormal_ret = round(stock_ret - spy_ret, 3)

        result = {
            "stock_return_pct":    stock_ret,
            "spy_return_pct":      spy_ret,
            "abnormal_return_pct": abnormal_ret,
            "window_start":        t0.strftime("%Y-%m-%d"),
            "window_end":          t_end.strftime("%Y-%m-%d"),
            "window_days":         window_days,
            "data_source":         "yahoo_finance",
            "market_reaction":     (
                "strong_positive" if abnormal_ret > 5  else
                "positive"        if abnormal_ret > 2  else
                "neutral"         if abnormal_ret > -2 else
                "negative"        if abnormal_ret > -5 else
                "strong_negative"
            ),
        }
        logger.debug(
            f"  Abnormal return {ticker} ({filing_date}): "
            f"stock={stock_ret:+.2f}% SPY={spy_ret:+.2f}% abnormal={abnormal_ret:+.2f}%"
        )
        return result

    except Exception as e:
        logger.debug(f"  yfinance error for {ticker}: {e}")
        return None


# ══════════════════════════════════════════════════════════════════════════════
# CAUSAL GUARD
# ══════════════════════════════════════════════════════════════════════════════

class CausalGuard:
    UNSUPPORTED_CAUSAL_PATTERNS = [
        r"\b(iphone|ipad|mac|airpods|services)\b.*\b(drove|driven|boosted|caused|led to)\b",
        r"\b(drove|driven|boosted|caused|led to)\b.*\b(revenue|growth|sales)\b",
        r"\bdue to (product|segment|customer|market) (demand|adoption|growth)\b",
        r"\b(strong|robust|high) demand (drove|boosted|caused)\b",
        r"\boperational efficiency (drove|caused|led to) (revenue|margin|profit)\b",
    ]
    _compiled = [re.compile(p, re.IGNORECASE) for p in UNSUPPORTED_CAUSAL_PATTERNS]

    MDA_DRIVER_PATTERNS = {
        "cost":          [r"cost(s)?\s+(increased|higher|rose|surge)", r"input cost", r"logistics cost", r"labor cost"],
        "demand":        [r"demand (increased|higher|strong|grew)", r"customer (demand|volume)", r"order volume"],
        "pricing":       [r"price (increase|higher)", r"pricing power", r"ASP"],
        "volume":        [r"unit (volume|sales|shipped)", r"shipment(s)?", r"units sold"],
        "fx":            [r"foreign (exchange|currency)", r"currency (headwind|tailwind|impact)"],
        "acquisition":   [r"acqui(red|sition|re)", r"merger", r"inorganic"],
        "restructuring": [r"restructur", r"headcount reduction", r"cost reduction program"],
        "macro":         [r"macroeconomic", r"recession", r"inflation", r"interest rate"],
    }
    _mda_compiled = {
        k: [re.compile(p, re.IGNORECASE) for p in pats]
        for k, pats in MDA_DRIVER_PATTERNS.items()
    }

    @classmethod
    def detect_drivers_in_mda(cls, mda_text: str) -> dict[str, bool]:
        if not mda_text:
            return {}
        return {
            driver: any(p.search(mda_text) for p in patterns)
            for driver, patterns in cls._mda_compiled.items()
        }

    @classmethod
    def validate_output(cls, output_text: str, mda_text: str) -> tuple[bool, list[str]]:
        if mda_text and len(mda_text) > 200:
            return True, []
        violations = []
        for pattern in cls._compiled:
            m = pattern.search(output_text)
            if m:
                violations.append(f"Unsupported causal claim: '{m.group()[:80]}'")
        return len(violations) == 0, violations

    @classmethod
    def build_causality_instruction(cls, metrics: dict, changes: dict, mda_text: str) -> str:
        has_mda = bool(mda_text and len(mda_text) > MIN_SECTION_CHARS)
        drivers = cls.detect_drivers_in_mda(mda_text) if has_mda else {}

        if has_mda and drivers:
            allowed = [k for k, v in drivers.items() if v]
            return (
                f"CAUSALITY RULE: MD&A text is available. You may attribute causal drivers "
                f"ONLY to the following evidence-backed themes found in MD&A: {', '.join(allowed) or 'none identified'}. "
                f"Do NOT invent product-level attribution unless the exact product/segment name "
                f"appears verbatim in the MD&A excerpt below."
            )
        elif has_mda and not drivers:
            return (
                "CAUSALITY RULE: MD&A available but no clear cost/demand/pricing drivers detected. "
                "Use metric-level causality only (e.g. 'gross margin compressed X pp'). "
                "Do NOT fabricate narrative drivers."
            )
        else:
            return (
                "CAUSALITY RULE: NO MD&A TEXT AVAILABLE. "
                "Base ALL causal statements STRICTLY on quantitative metrics. "
                "FORBIDDEN: any product name, segment name, demand narrative, or business unit reference. "
                "ALLOWED: 'Gross margin compressed Xpp', 'Operating leverage deteriorated', "
                "'R&D intensity at X% of revenue', 'FCF conversion at X% of net income'."
            )


# ══════════════════════════════════════════════════════════════════════════════
# MDA GROUNDER
# ══════════════════════════════════════════════════════════════════════════════

class MDAGrounder:
    ALIGNMENT_MAP = [
        ("margin_decline",   ["cost", "expense", "higher", "increase", "pressure"],
         "Margin decline consistent with MD&A references to elevated {mda_driver} costs."),
        ("margin_expansion", ["efficiency", "scale", "leverage", "reduction", "savings"],
         "Margin expansion supported by MD&A commentary on {mda_driver} improvements."),
        ("revenue_growth",   ["demand", "volume", "customer", "market", "growth"],
         "Revenue growth corroborated by MD&A references to {mda_driver}."),
        ("revenue_decline",  ["lower", "decline", "weak", "soft", "headwind"],
         "Revenue decline consistent with MD&A references to {mda_driver} headwinds."),
        ("capex_increase",   ["expansion", "invest", "capacity", "infrastructure", "build"],
         "CapEx increase aligned with MD&A investment narrative: {mda_driver}."),
        ("debt_increase",    ["acqui", "borrow", "financing", "credit", "facility"],
         "Leverage increase linked to MD&A financing activity: {mda_driver}."),
    ]

    @classmethod
    def _classify_metric_state(cls, changes: dict, metrics: dict) -> list[str]:
        states = []
        rev_d = changes.get("revenue_yoy_pct", 0) or 0
        gm_d  = changes.get("gross_margin_pct_delta", 0) or 0
        dbt_d = changes.get("debt_yoy_pct", 0) or 0
        if rev_d > 3:  states.append("revenue_growth")
        if rev_d < -3: states.append("revenue_decline")
        if gm_d < -1:  states.append("margin_decline")
        if gm_d > 1:   states.append("margin_expansion")
        if dbt_d > 10: states.append("debt_increase")
        return states

    @classmethod
    def build_grounded_context(cls, metrics: dict, changes: dict, mda_text: str) -> str:
        if not mda_text or len(mda_text) < MIN_SECTION_CHARS:
            return "[MD&A not available — quantitative causality only]"
        states  = cls._classify_metric_state(changes, metrics)
        mda_low = mda_text.lower()
        grounded = []
        for state in states:
            for condition, keywords, template in cls.ALIGNMENT_MAP:
                if condition != state:
                    continue
                found_kw = [kw for kw in keywords if kw in mda_low]
                if found_kw:
                    grounded.append(template.format(mda_driver=", ".join(found_kw[:3])))
                    break
        if not grounded:
            return "[MD&A available but no strong metric-narrative alignment detected — use quantitative causality]"
        return "MD&A ALIGNED CONTEXT (use these in your analysis):\n" + "\n".join(f"• {s}" for s in grounded)


# ══════════════════════════════════════════════════════════════════════════════
# EVIDENCE CONFIDENCE SCORER
# ══════════════════════════════════════════════════════════════════════════════

class EvidenceConfidence:
    @classmethod
    def score(cls, metrics: dict, changes: dict, text_sections: dict) -> dict:
        mda  = text_sections.get("mda", "") or ""
        risk = text_sections.get("risk_factors", "") or ""
        has_mda  = len(mda) > MIN_SECTION_CHARS
        has_risk = len(risk) > MIN_SECTION_CHARS

        key_metrics = ["revenue", "gross_profit", "operating_income", "net_income",
                       "free_cash_flow", "total_assets", "long_term_debt"]
        metric_coverage = sum(1 for m in key_metrics if m in metrics) / len(key_metrics)

        key_changes = ["revenue_yoy_pct", "gross_margin_pct_delta", "net_income_yoy_pct"]
        change_coverage = sum(1 for c in key_changes if c in changes) / len(key_changes)

        if metric_coverage >= 0.85 and change_coverage >= 0.8 and has_mda:
            level     = "HIGH"
            qualifier = ""
        elif metric_coverage >= 0.6 and change_coverage >= 0.5:
            level     = "MEDIUM"
            qualifier = "Note: Partial metric coverage — some conclusions may be limited."
        else:
            level     = "LOW"
            qualifier = "IMPORTANT: Limited data available. Conclusions are preliminary."

        return {
            "level":           level,
            "metric_coverage": round(metric_coverage, 2),
            "change_coverage": round(change_coverage, 2),
            "has_mda":         has_mda,
            "has_risk":        has_risk,
            "qualifier":       qualifier,
        }


# ══════════════════════════════════════════════════════════════════════════════
# QUALITY VALIDATION PIPELINE
# ══════════════════════════════════════════════════════════════════════════════

class QualityValidator:
    HALLUCINATION_PATTERNS = [
        r"\b(iphone|ipad|mac|services|aws|azure|ads)\b.{0,60}\b(drove|boosted|caused|led to)\b",
        r"\b(strong|robust|surging) demand\b",
        r"\boperational efficiency (drove|boosted|improved) (revenue|margin)\b",
        r"\bmarket (expansion|penetration|share gains) (drove|boosted|caused)\b",
        r"\b(product|customer|geographic) (mix|diversification) (drove|caused|boosted)\b",
    ]
    _hall_compiled   = [re.compile(p, re.IGNORECASE) for p in HALLUCINATION_PATTERNS]
    _filler_compiled = [re.compile(rf"\b{tok}\b", re.IGNORECASE) for tok in FILLER_TOKENS]

    @classmethod
    def check_hallucination(cls, output: str, has_mda: bool) -> dict:
        if has_mda:
            return {"flag": False, "violations": [], "note": "MD&A available — causal claims permitted"}
        violations = [p.search(output).group()[:80] for p in cls._hall_compiled if p.search(output)]
        return {
            "flag":       len(violations) > 0,
            "violations": violations,
            "note":       f"{len(violations)} unsupported causal claims detected" if violations else "Clean",
        }

    @classmethod
    def check_delta_consistency(cls, output: str, changes: dict) -> dict:
        issues = []
        for field, value in changes.items():
            if not field.endswith("_pct") or value is None:
                continue
            rounded = round(value, 1)
            pattern = (
                rf"({abs(rounded)-2:.0f}|{abs(rounded)-1:.0f}|{abs(rounded):.0f}"
                rf"|{abs(rounded)+1:.0f}|{abs(rounded)+2:.0f})\s*%"
            )
            if not re.search(pattern, output) and abs(value) > 3:
                issues.append(f"{field}={value:.1f}% not found in output")
        return {"issues": issues, "clean": len(issues) == 0}

    @classmethod
    def check_style(cls, output: str) -> dict:
        filler_hits = [p.pattern for p in cls._filler_compiled if p.search(output)]
        word_count  = len(output.split())
        num_count   = len(_NUM_RE.findall(output))
        density     = round(num_count / max(word_count, 1), 3)
        return {
            "filler_tokens":     filler_hits[:5],
            "word_count":        word_count,
            "numerical_density": density,
            "density_ok":        density >= 0.05,
            "length_ok":         80 <= word_count <= 350,
        }

    @classmethod
    def validate(cls, output: str, metrics: dict, changes: dict, has_mda: bool) -> dict:
        hall  = cls.check_hallucination(output, has_mda)
        delta = cls.check_delta_consistency(output, changes)
        style = cls.check_style(output)
        score = 100
        if hall["flag"]:            score -= 20 * len(hall["violations"])
        if not delta["clean"]:      score -= 5  * len(delta["issues"])
        if not style["density_ok"]: score -= 10
        if not style["length_ok"]:  score -= 5
        score = max(0, score)
        return {
            "quality_score":     score,
            "hallucination":     hall,
            "delta_consistency": delta,
            "style":             style,
            "passes_threshold":  score >= SILVER_QUALITY_THRESHOLD,
            "tier": (
                "gold"   if score >= GOLD_QUALITY_THRESHOLD   else
                "silver" if score >= SILVER_QUALITY_THRESHOLD else
                "bronze"
            ),
        }


# ══════════════════════════════════════════════════════════════════════════════
# STRUCTURAL RANDOMIZER
# ══════════════════════════════════════════════════════════════════════════════

class StructuralRandomizer:
    ORDERINGS = [
        ["revenue_analysis", "profitability", "profitability_drivers",
         "cash_flow", "balance_sheet", "key_risks", "investment_signal", "signal_rationale"],
        ["cash_flow", "revenue_analysis", "profitability", "profitability_drivers",
         "balance_sheet", "key_risks", "investment_signal", "signal_rationale"],
        ["revenue_analysis", "profitability", "key_risks", "cash_flow",
         "balance_sheet", "profitability_drivers", "investment_signal", "signal_rationale"],
        ["investment_signal", "signal_rationale", "revenue_analysis",
         "profitability", "profitability_drivers", "cash_flow", "balance_sheet", "key_risks"],
    ]

    @classmethod
    def get_ordering(cls, seed: Optional[str] = None) -> list[str]:
        if seed:
            idx = int(hashlib.md5(seed.encode()).hexdigest(), 16) % len(cls.ORDERINGS)
        else:
            idx = random.randint(0, len(cls.ORDERINGS) - 1)
        return cls.ORDERINGS[idx]

    @classmethod
    def build_output_schema_instruction(cls, ordering: list[str]) -> str:
        fields = []
        for k in ordering:
            if k == "key_risks":
                fields.append('  "key_risks": ["<risk1>", "<risk2>", "<risk3>"]')
            else:
                fields.append(f'  "{k}": "<...>"')
        return "{\n" + ",\n".join(fields) + "\n}"


# ─── UTILITY FUNCTIONS ────────────────────────────────────────────────────────

def normalize_financial_terms(text: str) -> str:
    for non_canonical, canonical in CANONICAL_TERMS.items():
        text = re.sub(re.escape(non_canonical), canonical, text, flags=re.IGNORECASE)
    return text


def _usd(v) -> str:
    if v is None: return "N/A"
    if abs(v) >= 1e12: return f"${v/1e12:.2f}T"
    if abs(v) >= 1e9:  return f"${v/1e9:.1f}B"
    if abs(v) >= 1e6:  return f"${v/1e6:.0f}M"
    return f"${v:,.0f}"

def _pct(v, sfx="%") -> str:
    return f"{v:+.1f}{sfx}" if v is not None else "N/A"

def _pct_plain(v) -> str:
    return f"{v:.1f}%" if v is not None else "N/A"


# ══════════════════════════════════════════════════════════════════════════════
# FIX-A: AGGRESSIVE TEXT EXTRACTION ENGINE
# ══════════════════════════════════════════════════════════════════════════════

def clean_section_text(raw: str) -> str:
    lines = raw.replace("\xa0", " ").replace("\u2019", "'").split("\n")
    seen, good = set(), []
    for raw_line in lines:
        line = raw_line.strip()
        if not line or len(line) < MIN_PARA_LEN:
            continue
        alpha = sum(1 for c in line if c.isalpha())
        if len(line) > 0 and alpha / len(line) < MIN_PARA_ALPHA:
            continue
        if line.isupper() and len(line) < 120:
            continue
        if re.search(r"\.\s*\.\s*\.\s*\d+\s*$", line):
            continue
        key = line[:80].lower()
        if key in seen:
            continue
        seen.add(key)
        good.append(line)
    return "\n\n".join(good)


def filter_financial_paragraphs(text: str, min_hits: int = 1) -> str:
    """FIX-A: Lowered min_hits from 2 to 1 for more aggressive extraction."""
    if not text:
        return ""
    paras    = [p.strip() for p in re.split(r"\n{2,}", text) if p.strip()]
    kept     = []
    for para in paras:
        lower    = para.lower()
        kw_hits  = sum(1 for kw in FINANCIAL_KEYWORDS if kw in lower)
        num_hits = len(_NUM_RE.findall(para))
        if kw_hits >= min_hits or num_hits >= 1:
            kept.append(para)
    return "\n\n".join(kept)[:MAX_TEXT_CHARS]


def score_text_quality(text: str) -> dict:
    """FIX-A: Score extracted text quality for logging and fallback decisions."""
    if not text:
        return {"chars": 0, "kw_hits": 0, "num_hits": 0, "score": 0, "grade": "empty"}
    lower    = text.lower()
    kw_hits  = sum(1 for kw in FINANCIAL_KEYWORDS if kw in lower)
    num_hits = len(_NUM_RE.findall(text))
    score    = kw_hits * 2 + num_hits
    grade    = "excellent" if score > 30 else "good" if score > 15 else "fair" if score > 5 else "poor"
    return {"chars": len(text), "kw_hits": kw_hits, "num_hits": num_hits, "score": score, "grade": grade}


def extract_section(text: str, patterns: list[str], doc_len: int) -> str:
    skip_start = max(3000, int(doc_len * 0.03))  # FIX-A: reduced skip margin
    lower      = text.lower()

    def _find(sf: int) -> str:
        start = -1
        for pat in patterns:
            m = re.search(pat, lower[sf:], re.DOTALL)
            if m:
                start = sf + m.start()
                break
        if start < 0:
            return ""
        tail  = lower[start + 30:]
        end_m = NEXT_ITEM_RE.search(tail)
        end   = (start + 30 + end_m.start()) if end_m else min(start + MAX_TEXT_CHARS * 2, len(text))
        raw   = text[start:end].strip()
        return filter_financial_paragraphs(clean_section_text(raw), min_hits=1)

    r1 = _find(skip_start)
    if r1 and len(r1) >= MIN_SECTION_CHARS:
        return r1[:MAX_TEXT_CHARS]
    r2 = _find(0)
    return (r2 if len(r2) > len(r1) else r1)[:MAX_TEXT_CHARS]


def extract_section_fallback(text: str, fallback_patterns: list) -> str:
    """
    FIX-A Strategy 2: Fallback extraction using loose semantic patterns.
    Scans document for any paragraph matching financial content keywords.
    """
    lower  = text.lower()
    found_positions = []
    for pat in fallback_patterns:
        for m in pat.finditer(lower):
            found_positions.append(m.start())

    if not found_positions:
        return ""

    # Take best position (most keyword hits in surrounding context)
    best_pos   = -1
    best_score = -1
    for pos in found_positions[:10]:
        snippet = text[max(0, pos-100): pos + 3000]
        sc      = score_text_quality(snippet)["score"]
        if sc > best_score:
            best_score = sc
            best_pos   = pos

    if best_pos < 0:
        return ""

    raw = text[max(0, best_pos - 50): best_pos + MAX_TEXT_CHARS]
    return filter_financial_paragraphs(clean_section_text(raw), min_hits=1)[:MAX_TEXT_CHARS]


def extract_by_density_scan(text: str, window_size: int = 3000, top_n: int = 3) -> str:
    """
    FIX-A Strategy 3: Sliding window density scan.
    Find the N windows with highest financial keyword + number density.
    """
    if not text or len(text) < window_size:
        return filter_financial_paragraphs(clean_section_text(text), min_hits=1)

    step         = window_size // 2
    best_windows = []
    for i in range(0, len(text) - window_size, step):
        window = text[i: i + window_size]
        sc     = score_text_quality(window)["score"]
        best_windows.append((sc, i))

    best_windows.sort(reverse=True)
    top_windows = sorted(best_windows[:top_n], key=lambda x: x[1])  # restore order

    combined = "\n\n".join(text[i: i + window_size] for _, i in top_windows)
    return filter_financial_paragraphs(clean_section_text(combined), min_hits=1)[:MAX_TEXT_CHARS]


_RE_IX_HIDDEN = re.compile(r"<ix:hidden[^>]*>.*?</ix:hidden\s*>", re.DOTALL | re.IGNORECASE)
_RE_IX_TAGS   = re.compile(r"</?ix:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)
_RE_XBRL_TAGS = re.compile(r"</?xbrli?:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)
_RE_LINK_TAGS = re.compile(r"</?link:[A-Za-z][A-Za-z0-9]*(?:\s[^>]*)?>", re.IGNORECASE)


def preprocess_ixbrl(html: str) -> str:
    html = _RE_IX_HIDDEN.sub(" ", html)
    html = _RE_IX_TAGS.sub("",   html)
    html = _RE_XBRL_TAGS.sub("", html)
    html = _RE_LINK_TAGS.sub("", html)
    return html


def html_to_text(html: str) -> str:
    try:
        html = preprocess_ixbrl(html)
        soup = BeautifulSoup(html, "lxml")
        for tag in soup(["script", "style", "head", "noscript", "meta", "link"]):
            tag.decompose()
        text = soup.get_text(separator="\n")
        text = re.sub(r"[ \t]{2,}", " ", text)
        text = re.sub(r"\n{3,}", "\n\n", text)
        return text.strip()
    except Exception as e:
        logger.warning(f"html_to_text fallback: {e}")
        html = _RE_IX_HIDDEN.sub(" ", html)
        text = re.sub(r"<[^>]+>", " ", html)
        text = re.sub(r"[ \t]{2,}", " ", text)
        return re.sub(r"\n{3,}", "\n\n", text).strip()


def _is_xbrl_json(text: str) -> bool:
    s = text.strip()
    return s.startswith("{") and (
        '"nsprefix"' in s[:200] or
        ('"version"' in s[:100] and '"instance"' in s[:200])
    )


def _is_xbrl_stub(fname: str) -> bool:
    return bool(re.match(r"^R\d+\.htm$", fname, re.IGNORECASE))


def _item_score(plain: str) -> int:
    return len(ITEM_HEADING_RE.findall(plain))


# ─── TICKERS ──────────────────────────────────────────────────────────────────

def get_nasdaq100_tickers() -> list[dict]:
    return [
        {"ticker": "AAPL",  "name": "Apple Inc"},
        {"ticker": "MSFT",  "name": "Microsoft Corp"},
        {"ticker": "NVDA",  "name": "NVIDIA Corp"},
        {"ticker": "GOOGL", "name": "Alphabet Inc"},
        {"ticker": "AMZN",  "name": "Amazon.com Inc"},
        {"ticker": "META",  "name": "Meta Platforms Inc"},
        {"ticker": "TSLA",  "name": "Tesla Inc"},
        {"ticker": "AVGO",  "name": "Broadcom Inc"},
        {"ticker": "COST",  "name": "Costco Wholesale Corp"},
        {"ticker": "NFLX",  "name": "Netflix Inc"},
        {"ticker": "AMD",   "name": "Advanced Micro Devices"},
        {"ticker": "QCOM",  "name": "Qualcomm Inc"},
        {"ticker": "ADBE",  "name": "Adobe Inc"},
        {"ticker": "AMAT",  "name": "Applied Materials Inc"},
        {"ticker": "INTU",  "name": "Intuit Inc"},
        {"ticker": "MU",    "name": "Micron Technology Inc"},
        {"ticker": "LRCX",  "name": "Lam Research Corp"},
        {"ticker": "KLAC",  "name": "KLA Corp"},
        {"ticker": "PANW",  "name": "Palo Alto Networks"},
        {"ticker": "CRWD",  "name": "CrowdStrike Holdings"},
        {"ticker": "CSCO",  "name": "Cisco Systems Inc"},
        {"ticker": "PEP",   "name": "PepsiCo Inc"},
        {"ticker": "TMUS",  "name": "T-Mobile US Inc"},
        {"ticker": "ADP",   "name": "Automatic Data Processing"},
        {"ticker": "SBUX",  "name": "Starbucks Corp"},
        {"ticker": "GILD",  "name": "Gilead Sciences Inc"},
        {"ticker": "BKNG",  "name": "Booking Holdings Inc"},
        {"ticker": "ISRG",  "name": "Intuitive Surgical Inc"},
        {"ticker": "AMGN",  "name": "Amgen Inc"},
        {"ticker": "ADI",   "name": "Analog Devices Inc"},
        {"ticker": "TXN",   "name": "Texas Instruments Inc"},
        {"ticker": "MELI",  "name": "MercadoLibre Inc"},
        {"ticker": "PYPL",  "name": "PayPal Holdings Inc"},
        {"ticker": "ABNB",  "name": "Airbnb Inc"},
        {"ticker": "DASH",  "name": "DoorDash Inc"},
        {"ticker": "FTNT",  "name": "Fortinet Inc"},
        {"ticker": "NET",   "name": "Cloudflare Inc"},
        {"ticker": "MNST",  "name": "Monster Beverage Corp"},
        {"ticker": "ORLY",  "name": "O'Reilly Automotive Inc"},
        {"ticker": "PAYX",  "name": "Paychex Inc"},
        {"ticker": "PCAR",  "name": "PACCAR Inc"},
        {"ticker": "SPOT",  "name": "Spotify Technology SA"},
        {"ticker": "TSCO",  "name": "Tractor Supply Co"},
        {"ticker": "DLTR",  "name": "Dollar Tree Inc"},
        {"ticker": "EA",    "name": "Electronic Arts Inc"},
        {"ticker": "EBAY",  "name": "eBay Inc"},
        {"ticker": "FAST",  "name": "Fastenal Co"},
        {"ticker": "FISV",  "name": "Fiserv Inc"},
        {"ticker": "CDW",   "name": "CDW Corp"},
        {"ticker": "CEG",   "name": "Constellation Energy Corp"},
        {"ticker": "CDNS",  "name": "Cadence Design Systems"},
        {"ticker": "SNPS",  "name": "Synopsys Inc"},
        {"ticker": "MRVL",  "name": "Marvell Technology Inc"},
        {"ticker": "ROST",  "name": "Ross Stores Inc"},
        {"ticker": "ODFL",  "name": "Old Dominion Freight Line"},
        {"ticker": "CTAS",  "name": "Cintas Corp"},
        {"ticker": "VRSK",  "name": "Verisk Analytics Inc"},
        {"ticker": "CPRT",  "name": "Copart Inc"},
        {"ticker": "IDXX",  "name": "IDEXX Laboratories Inc"},
        {"ticker": "CTSH",  "name": "Cognizant Technology Solutions"},
        {"ticker": "DXCM",  "name": "DexCom Inc"},
        {"ticker": "BIIB",  "name": "Biogen Inc"},
        {"ticker": "ILMN",  "name": "Illumina Inc"},
        {"ticker": "REGN",  "name": "Regeneron Pharmaceuticals"},
        {"ticker": "VRTX",  "name": "Vertex Pharmaceuticals"},
        {"ticker": "MRNA",  "name": "Moderna Inc"},
        {"ticker": "ZS",    "name": "Zscaler Inc"},
        {"ticker": "OKTA",  "name": "Okta Inc"},
        {"ticker": "SNOW",  "name": "Snowflake Inc"},
        {"ticker": "DDOG",  "name": "Datadog Inc"},
        {"ticker": "MDB",   "name": "MongoDB Inc"},
        {"ticker": "TTD",   "name": "The Trade Desk Inc"},
        {"ticker": "TEAM",  "name": "Atlassian Corp"},
        {"ticker": "WDAY",  "name": "Workday Inc"},
        {"ticker": "VEEV",  "name": "Veeva Systems Inc"},
        {"ticker": "NOW",   "name": "ServiceNow Inc"},
        {"ticker": "CRM",   "name": "Salesforce Inc"},
        {"ticker": "ORCL",  "name": "Oracle Corp"},
        {"ticker": "IBM",   "name": "IBM Corp"},
        {"ticker": "HPQ",   "name": "HP Inc"},
        {"ticker": "DELL",  "name": "Dell Technologies Inc"},
        {"ticker": "STX",   "name": "Seagate Technology Holdings"},
        {"ticker": "WDC",   "name": "Western Digital Corp"},
        {"ticker": "NTAP",  "name": "NetApp Inc"},
        {"ticker": "KEYS",  "name": "Keysight Technologies Inc"},
        {"ticker": "NXPI",  "name": "NXP Semiconductors NV"},
        {"ticker": "MCHP",  "name": "Microchip Technology Inc"},
        {"ticker": "ON",    "name": "ON Semiconductor Corp"},
        {"ticker": "SWKS",  "name": "Skyworks Solutions Inc"},
        {"ticker": "MPWR",  "name": "Monolithic Power Systems"},
        {"ticker": "ENPH",  "name": "Enphase Energy Inc"},
        {"ticker": "ZM",    "name": "Zoom Video Communications"},
        {"ticker": "DOCU",  "name": "DocuSign Inc"},
        {"ticker": "ALGN",  "name": "Align Technology Inc"},
    ]


# ─── HTTP SESSION ─────────────────────────────────────────────────────────────

class EDGARSession:
    ARCHIVES = "https://www.sec.gov/Archives/edgar/data"
    DATA_BASE = "https://data.sec.gov"
    WWW_BASE  = "https://www.sec.gov"

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update(EDGAR_HEADERS)

    def get(self, url: str, timeout: int = 30, is_html: bool = False) -> Optional[requests.Response]:
        time.sleep(EDGAR_SLEEP)
        hdrs = {**EDGAR_HEADERS, "Accept": "text/html,*/*;q=0.8"} if is_html else EDGAR_HEADERS
        for attempt in range(MAX_RETRIES):
            try:
                r = self.session.get(url, timeout=timeout, headers=hdrs)
                if r.status_code == 429:
                    wait = _rate_limiter.jitter(BACKOFF_BASE ** (attempt + 2))
                    wait = min(wait, BACKOFF_MAX)
                    logger.warning(f"EDGAR 429 → sleep {wait:.1f}s")
                    time.sleep(wait)
                    continue
                if r.status_code in (403, 404):
                    logger.debug(f"HTTP {r.status_code}: {url[-70:]}")
                    return None
                r.raise_for_status()
                return r
            except requests.exceptions.Timeout:
                wait = _rate_limiter.jitter(BACKOFF_BASE ** (attempt + 1))
                time.sleep(min(wait, BACKOFF_MAX))
            except Exception as e:
                logger.error(f"GET: {e} | {url[-60:]}")
                return None
        return None


# ─── CIK LOOKUP ───────────────────────────────────────────────────────────────

_cik_cache:    dict[str, str]   = {}
_tickers_data: Optional[dict]   = None


def ticker_to_cik(ticker: str, sess: EDGARSession) -> Optional[str]:
    global _tickers_data
    if ticker in _cik_cache:
        return _cik_cache[ticker]
    if _tickers_data is None:
        r = sess.get(f"{sess.WWW_BASE}/files/company_tickers.json")
        _tickers_data = r.json() if r else {}
        logger.info(f"CIK table: {len(_tickers_data):,} entries")
    for entry in _tickers_data.values():
        if entry.get("ticker", "").upper() == ticker.upper():
            cik = str(entry["cik_str"]).zfill(10)
            _cik_cache[ticker] = cik
            return cik
    logger.warning(f"No CIK for {ticker}")
    return None


# ─── FILING DISCOVERY ─────────────────────────────────────────────────────────

def get_filings(cik: str, form_type: str, sess: EDGARSession, max_results: int = 10) -> list[dict]:
    url = f"{sess.DATA_BASE}/submissions/CIK{cik}.json"
    r   = sess.get(url)
    if not r:
        return []
    data   = r.json()
    recent = data.get("filings", {}).get("recent", {})
    cutoff = (datetime.now() - timedelta(days=YEARS_BACK * 365)).strftime("%Y-%m-%d")

    rows = list(zip(
        recent.get("form", []), recent.get("filingDate", []),
        recent.get("reportDate", []), recent.get("accessionNumber", []),
        recent.get("primaryDocument", []),
    ))
    for ef in data.get("filings", {}).get("files", [])[:5]:
        fname = ef.get("name", "")
        if not fname:
            continue
        r2 = sess.get(f"{sess.DATA_BASE}/submissions/{fname}")
        if not r2:
            continue
        try:
            ed = r2.json()
            rows += list(zip(
                ed.get("form", []), ed.get("filingDate", []),
                ed.get("reportDate", []), ed.get("accessionNumber", []),
                ed.get("primaryDocument", []),
            ))
        except Exception:
            pass

    base    = form_type.split("/")[0]
    results = []
    for form, dt, period, accno, doc in rows:
        if base not in form:
            continue
        if dt < cutoff:
            continue
        results.append({
            "form_type": form, "filed_date": dt, "period_of_report": period,
            "accession_no": accno, "primary_document": doc,
        })
        if len(results) >= max_results:
            break

    results.reverse()
    logger.info(f"  {form_type}: {len(results)} filings")
    return results


# ─── DOCUMENT SELECTION ───────────────────────────────────────────────────────

def _get_index_docs(cik: str, accession_no: str, sess: EDGARSession) -> list[dict]:
    clean = accession_no.replace("-", "")
    r = sess.get(f"{sess.ARCHIVES}/{int(cik)}/{clean}/{accession_no}-index.json")
    if not r:
        return []
    try:
        return r.json().get("documents", [])
    except Exception:
        return []


def get_narrative_doc(
    cik: str, accession_no: str, primary_doc: str, sess: EDGARSession
) -> tuple[Optional[str], str]:
    clean = accession_no.replace("-", "")
    base  = f"{sess.ARCHIVES}/{int(cik)}/{clean}"

    fp_result = None
    if primary_doc and primary_doc.lower().endswith(".htm"):
        url = f"{base}/{primary_doc}"
        r   = sess.get(url, timeout=120, is_html=True)
        if r and len(r.content) > MIN_NARRATIVE_DOC_BYTES:
            plain = html_to_text(r.text)
            score = _item_score(plain)
            logger.debug(f"  Fast-path ({len(r.content)//1024}KB, score={score}): {primary_doc}")
            if score >= 3:   # FIX-A: lowered from 5
                return url, r.text
            fp_result = (url, r.text, score)

    docs = _get_index_docs(cik, accession_no, sess)
    candidates = [
        doc.get("document", "") for doc in docs
        if doc.get("document", "").lower().endswith(".htm")
        and not _is_xbrl_stub(doc.get("document", ""))
        and not any(p in doc.get("document", "").lower() for p in ["cover", "signature"])
    ]

    if not candidates:
        if fp_result:
            return fp_result[0], fp_result[1]
        logger.warning(f"  No candidates for {accession_no}")
        return None, ""

    best_url, best_html, best_score = "", "", -1
    for dfile in candidates[:8]:   # FIX-A: check more candidates
        url = f"{base}/{dfile}"
        r   = sess.get(url, timeout=120, is_html=True)
        if not r or len(r.content) < 5_000:  # FIX-A: lowered threshold
            continue
        plain = html_to_text(r.text)
        score = _item_score(plain)
        logger.debug(f"    Candidate score={score} ({len(r.content)//1024}KB): {dfile}")
        if score > best_score:
            best_score = score
            best_url   = url
            best_html  = r.text

    if best_score >= 1:   # FIX-A: accept any doc with at least 1 item heading
        logger.info(f"  Doc (score={best_score}): {best_url.split('/')[-1]}")
        return best_url, best_html

    if fp_result:
        return fp_result[0], fp_result[1]
    logger.warning(f"  No narrative doc for {accession_no}")
    return None, ""


def get_8k_content(cik: str, accession_no: str, primary_doc: str, sess: EDGARSession) -> str:
    clean = accession_no.replace("-", "")
    base  = f"{sess.ARCHIVES}/{int(cik)}/{clean}"
    docs  = _get_index_docs(cik, accession_no, sess)

    for doc in docs:
        dtype = doc.get("type", "").upper()
        dfile = doc.get("document", "")
        if "99" in dtype and (dfile.lower().endswith(".htm") or dfile.lower().endswith(".txt")):
            r = sess.get(f"{base}/{dfile}", timeout=60, is_html=True)
            if r and len(r.content) > 1_000:
                text = html_to_text(r.text)
                if _is_xbrl_json(text):
                    continue
                if sum(1 for kw in MATERIAL_8K_KEYWORDS if kw in text.lower()) >= 2:
                    return text[:4000]

    for doc in docs:
        dfile = doc.get("document", "")
        if not dfile.lower().endswith(".htm") or _is_xbrl_stub(dfile):
            continue
        r = sess.get(f"{base}/{dfile}", timeout=60, is_html=True)
        if r and len(r.content) > 800:
            text = html_to_text(r.text)
            if _is_xbrl_json(text):
                continue
            if sum(1 for kw in MATERIAL_8K_KEYWORDS if kw in text.lower()) >= 2:
                return text[:4000]

    r_txt = sess.get(f"{base}/{accession_no}.txt", timeout=60)
    if r_txt:
        best = ""
        for block in re.findall(r"<DOCUMENT>(.*?)</DOCUMENT>", r_txt.text, re.DOTALL | re.IGNORECASE):
            tm      = re.search(r"<TEXT>(.*?)</TEXT>", block, re.DOTALL | re.IGNORECASE)
            content = tm.group(1) if tm else block
            content = html_to_text(content) if "<" in content else re.sub(r"\s{3,}", "\n\n", content).strip()
            if _is_xbrl_json(content):
                continue
            kw = sum(1 for k in MATERIAL_8K_KEYWORDS if k in content.lower())
            if kw > 2 and len(content) > len(best):
                best = content
        if best:
            return best[:4000]

    logger.warning(f"  8-K no content: {accession_no}")
    return ""


def is_material_8k(text: str) -> bool:
    if not text or len(text) < 200:
        return False
    return sum(1 for kw in MATERIAL_8K_KEYWORDS if kw in text.lower()) >= 3


def scrape_text_sections(
    cik: str, accession_no: str, primary_doc: str, form_type: str, sess: EDGARSession
) -> dict[str, str]:
    """
    FIX-A: Multi-strategy text extraction.
    Strategy 1: Standard item-heading pattern matching
    Strategy 2: Fallback semantic patterns (MDA_FALLBACK_RE, RISK_FALLBACK_RE)
    Strategy 3: Density scan on full document
    Strategy 4: SGML full-text fallback
    """
    url, html = get_narrative_doc(cik, accession_no, primary_doc, sess)
    result    = {}

    if not html:
        logger.warning(f"  No HTML for {accession_no} — trying SGML")
        html = _sgml_fallback(cik, accession_no, sess)
        if not html:
            return {}

    plain      = html_to_text(html)
    score      = _item_score(plain)
    section_map = SECTIONS_10K if "10-K" in form_type else SECTIONS_10Q

    # Strategy 1: Standard patterns
    for name, pats in section_map.items():
        result[name] = extract_section(plain, pats, len(plain))

    filled = sum(1 for v in result.values() if v and len(v) >= MIN_SECTION_CHARS)
    logger.info(f"    Strategy-1 sections: {filled}/{len(section_map)} (item_score={score})")

    # Strategy 2: Fallback patterns for empty critical sections
    if not result.get("mda") or len(result.get("mda", "")) < MIN_SECTION_CHARS:
        fb = extract_section_fallback(plain, MDA_FALLBACK_RE)
        if fb and len(fb) >= MIN_SECTION_CHARS:
            result["mda"] = fb
            logger.info(f"    Strategy-2 MD&A fallback: {len(fb)} chars")

    if not result.get("risk_factors") or len(result.get("risk_factors", "")) < MIN_SECTION_CHARS:
        fb = extract_section_fallback(plain, RISK_FALLBACK_RE)
        if fb and len(fb) >= MIN_SECTION_CHARS:
            result["risk_factors"] = fb
            logger.info(f"    Strategy-2 Risk fallback: {len(fb)} chars")

    # Strategy 3: Density scan if still empty
    filled_now = sum(1 for v in result.values() if v and len(v) >= MIN_SECTION_CHARS)
    if filled_now == 0 and len(plain) > 2000:
        density_text = extract_by_density_scan(plain)
        if density_text and len(density_text) >= MIN_SECTION_CHARS:
            result["mda"] = density_text
            logger.info(f"    Strategy-3 density scan: {len(density_text)} chars")

    filled_final = sum(1 for v in result.values() if v and len(v) >= MIN_SECTION_CHARS)
    text_quality = score_text_quality(result.get("mda", "") or "")
    logger.info(
        f"    Final sections: {filled_final}/{len(section_map)} | "
        f"MD&A quality: {text_quality['grade']} ({text_quality['chars']} chars)"
    )
    return result


def _sgml_fallback(cik: str, accession_no: str, sess: EDGARSession) -> str:
    """FIX-A Strategy 4: Parse the SGML complete submission file."""
    clean = accession_no.replace("-", "")
    base  = f"{sess.ARCHIVES}/{int(cik)}/{clean}"
    r_txt = sess.get(f"{base}/{accession_no}.txt", timeout=90)
    if not r_txt:
        return ""
    best = ""
    for block in re.findall(r"<DOCUMENT>(.*?)</DOCUMENT>", r_txt.text, re.DOTALL | re.IGNORECASE):
        tm      = re.search(r"<TEXT>(.*?)</TEXT>", block, re.DOTALL | re.IGNORECASE)
        content = tm.group(1) if tm else block
        content = html_to_text(content) if "<" in content else re.sub(r"\s{3,}", "\n\n", content).strip()
        if _is_xbrl_json(content):
            continue
        sq = score_text_quality(content)
        if sq["score"] > score_text_quality(best)["score"]:
            best = content
    return best


# ─── XBRL METRICS ─────────────────────────────────────────────────────────────

FIELD_MAP = {
    "revenue":                   ["RevenueFromContractWithCustomerExcludingAssessedTax",
                                  "NetSales", "RevenueFromContractWithCustomerIncludingAssessedTax",
                                  "Revenues", "SalesRevenueNet"],
    "cost_of_revenue":           ["CostOfGoodsAndServicesSold", "CostOfRevenue", "CostOfGoodsSold"],
    "gross_profit":              ["GrossProfit"],
    "operating_expenses":        ["OperatingExpenses"],
    "research_development":      ["ResearchAndDevelopmentExpense"],
    "selling_general_admin":     ["SellingGeneralAndAdministrativeExpense"],
    "operating_income":          ["OperatingIncomeLoss"],
    "interest_expense":          ["InterestExpense", "InterestAndDebtExpense"],
    "pretax_income":             ["IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest"],
    "income_tax":                ["IncomeTaxExpenseBenefit"],
    "net_income":                ["NetIncomeLoss"],
    "eps_basic":                 ["EarningsPerShareBasic"],
    "eps_diluted":               ["EarningsPerShareDiluted"],
    "shares_diluted":            ["WeightedAverageNumberOfDilutedSharesOutstanding"],
    "total_assets":              ["Assets"],
    "current_assets":            ["AssetsCurrent"],
    "total_liabilities":         ["Liabilities"],
    "current_liabilities":       ["LiabilitiesCurrent"],
    "total_equity":              ["StockholdersEquity",
                                  "StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest"],
    "cash_and_equivalents":      ["CashAndCashEquivalentsAtCarryingValue",
                                  "CashCashEquivalentsAndShortTermInvestments",
                                  "CashCashEquivalentsRestrictedCashAndRestrictedCashEquivalents"],
    "short_term_investments":    ["ShortTermInvestments", "MarketableSecuritiesCurrent"],
    "accounts_receivable":       ["AccountsReceivableNetCurrent"],
    "inventory":                 ["InventoryNet"],
    "goodwill":                  ["Goodwill"],
    "long_term_debt":            ["LongTermDebt", "LongTermDebtNoncurrent"],
    "retained_earnings":         ["RetainedEarningsAccumulatedDeficit"],
    "operating_cash_flow":       ["NetCashProvidedByUsedInOperatingActivities"],
    "investing_cash_flow":       ["NetCashProvidedByUsedInInvestingActivities"],
    "financing_cash_flow":       ["NetCashProvidedByUsedInFinancingActivities"],
    "capex":                     ["PaymentsToAcquirePropertyPlantAndEquipment"],
    "depreciation_amortization": ["DepreciationDepletionAndAmortization", "DepreciationAndAmortization"],
    "stock_based_compensation":  ["ShareBasedCompensation", "AllocatedShareBasedCompensationExpense"],
    "dividends_paid":            ["PaymentsOfDividends", "PaymentsOfDividendsCommonStock"],
    "share_repurchases":         ["PaymentsForRepurchaseOfCommonStock"],
}
DEI_MAP = {
    "employees":           ["EntityNumberOfEmployees"],
    "entity_public_float": ["EntityPublicFloat"],
}


def load_xbrl_facts(cik: str, sess: EDGARSession) -> dict:
    r = sess.get(f"{sess.DATA_BASE}/api/xbrl/companyfacts/CIK{cik}.json", timeout=30)
    if not r:
        return {}
    facts = r.json().get("facts", {})
    gaap  = facts.get("us-gaap", {})
    dei   = facts.get("dei", {})
    logger.info(f"  XBRL: {len(gaap)} GAAP + {len(dei)} DEI")
    return {"us-gaap": gaap, "dei": dei}


def _is_consolidated(e: dict) -> bool:
    return e.get("segment") is None


def xbrl_val(ns: dict, concepts: list, period_end: str, base_form: str) -> Optional[float]:
    try:
        td     = datetime.strptime(period_end, "%Y-%m-%d")
        window = {
            (td + timedelta(days=d)).strftime("%Y-%m-%d")
            for d in range(-PERIOD_FUZZ_DAYS, PERIOD_FUZZ_DAYS + 1)
        }
    except Exception:
        window = {period_end}
    for concept in concepts:
        for unit in ("USD", "USD/shares", "shares", "pure"):
            matches = [
                e for e in ns.get(concept, {}).get("units", {}).get(unit, [])
                if e.get("end") in window
                and e.get("form", "").startswith(base_form)
                and e.get("val") is not None
                and _is_consolidated(e)
            ]
            if matches:
                matches.sort(key=lambda e: e.get("filed", ""), reverse=True)
                try:
                    return float(matches[0]["val"])
                except (ValueError, TypeError):
                    pass
    return None


def extract_metrics(xbrl: dict, period_end: str, form_type: str) -> dict:
    gaap = xbrl.get("us-gaap", {})
    dei  = xbrl.get("dei", {})
    bf   = form_type.split("/")[0]
    m    = {}
    for field, concepts in FIELD_MAP.items():
        m[field] = xbrl_val(gaap, concepts, period_end, bf)
    for field, concepts in DEI_MAP.items():
        m[field] = xbrl_val(dei, concepts, period_end, bf)

    ocf   = m.get("operating_cash_flow")
    capex = m.get("capex")
    if ocf is not None and capex is not None:
        m["free_cash_flow"] = ocf - abs(capex)
    oi = m.get("operating_income")
    da = m.get("depreciation_amortization")
    if oi is not None and da is not None:
        m["ebitda"] = oi + abs(da)

    rev  = m.get("revenue")
    gp   = m.get("gross_profit")
    ni   = m.get("net_income")
    ta   = m.get("total_assets")
    eq   = m.get("total_equity")
    dbt  = m.get("long_term_debt")
    csh  = m.get("cash_and_equivalents")
    oi2  = m.get("operating_income")

    if rev and rev > 0:
        if gp  is not None: m["gross_margin_pct"]     = round(gp  / rev * 100, 2)
        if ni  is not None: m["net_margin_pct"]        = round(ni  / rev * 100, 2)
        if oi2 is not None: m["operating_margin_pct"]  = round(oi2 / rev * 100, 2)
        cor = m.get("cost_of_revenue")
        sga = m.get("selling_general_admin")
        rnd = m.get("research_development")
        if cor is not None: m["cogs_ratio_pct"] = round(cor / rev * 100, 2)
        if sga is not None: m["sga_ratio_pct"]  = round(sga / rev * 100, 2)
        if rnd is not None: m["rd_ratio_pct"]   = round(rnd / rev * 100, 2)

    if ta and ta > 0 and ni is not None:
        m["return_on_assets_pct"] = round(ni / ta * 100, 2)
    if eq and eq > 0 and ni is not None:
        m["return_on_equity_pct"] = round(ni / eq * 100, 2)
    if dbt is not None and csh is not None:
        m["net_debt"] = dbt - csh
    if eq and eq > 0 and dbt is not None:
        m["debt_to_equity"] = round(dbt / eq, 3)
    ca = m.get("current_assets")
    cl = m.get("current_liabilities")
    if ca and cl and cl > 0:
        m["current_ratio"] = round(ca / cl, 3)

    ni_v  = m.get("net_income")
    fcf_v = m.get("free_cash_flow")
    if ni_v and ni_v > 0 and fcf_v is not None:
        m["fcf_conversion_ratio"] = round(fcf_v / ni_v, 3)

    return {
        k: (round(v, 4) if isinstance(v, float) and abs(v) > 0.01 else v)
        for k, v in m.items()
        if v is not None
    }


# ─── FIX-1: DE-CUMULATION ─────────────────────────────────────────────────────

def get_fiscal_quarter(period_end: str, form_type: str) -> Optional[int]:
    if not period_end:
        return None
    try:
        dt = datetime.strptime(period_end[:10], "%Y-%m-%d")
    except ValueError:
        return None
    if "10-K" in form_type:
        return 4
    month = dt.month
    if month in (1, 2, 3, 4):   return 1
    elif month in (4, 5, 6, 7): return 2
    elif month in (7, 8, 9, 10): return 3
    else:                        return 4


def decumulate_10q_metrics(
    current_metrics: dict,
    prev_ytd_metrics: dict,
    fiscal_quarter: int,
    form_type: str,
) -> tuple[dict, bool]:
    if "10-Q" not in form_type or fiscal_quarter == 1 or not prev_ytd_metrics:
        return current_metrics, False

    FLOW_METRICS = {
        "revenue", "cost_of_revenue", "gross_profit", "operating_expenses",
        "research_development", "selling_general_admin", "operating_income",
        "interest_expense", "pretax_income", "income_tax", "net_income",
        "depreciation_amortization", "stock_based_compensation",
        "operating_cash_flow", "investing_cash_flow", "financing_cash_flow",
        "capex", "dividends_paid", "share_repurchases",
    }

    discrete = dict(current_metrics)
    applied  = False

    for field in FLOW_METRICS:
        cur  = current_metrics.get(field)
        prev = prev_ytd_metrics.get(field)
        if cur is None or prev is None:
            continue
        discrete[field] = round(cur - prev, 4)
        applied = True

    rev = discrete.get("revenue")
    gp  = discrete.get("gross_profit")
    ni  = discrete.get("net_income")
    oi  = discrete.get("operating_income")
    da  = discrete.get("depreciation_amortization")
    ocf = discrete.get("operating_cash_flow")
    cap = discrete.get("capex")

    if rev and rev > 0:
        if gp  is not None: discrete["gross_margin_pct"]     = round(gp  / rev * 100, 2)
        if ni  is not None: discrete["net_margin_pct"]       = round(ni  / rev * 100, 2)
        if oi  is not None: discrete["operating_margin_pct"] = round(oi  / rev * 100, 2)
        cor = discrete.get("cost_of_revenue")
        sga = discrete.get("selling_general_admin")
        rnd = discrete.get("research_development")
        if cor is not None: discrete["cogs_ratio_pct"] = round(cor / rev * 100, 2)
        if sga is not None: discrete["sga_ratio_pct"]  = round(sga / rev * 100, 2)
        if rnd is not None: discrete["rd_ratio_pct"]   = round(rnd / rev * 100, 2)

    if oi is not None and da is not None:
        discrete["ebitda"] = oi + abs(da)
    if ocf is not None and cap is not None:
        discrete["free_cash_flow"] = ocf - abs(cap)
    ni_v  = discrete.get("net_income")
    fcf_v = discrete.get("free_cash_flow")
    if ni_v and ni_v > 0 and fcf_v is not None:
        discrete["fcf_conversion_ratio"] = round(fcf_v / ni_v, 3)

    logger.debug(
        f"    De-cumulate Q{fiscal_quarter}: "
        f"rev {_usd(current_metrics.get('revenue'))} YTD → {_usd(discrete.get('revenue'))} discrete"
    )
    return discrete, applied


def extract_metrics_with_decumulation(
    xbrl: dict,
    period_end: str,
    form_type: str,
    prev_ytd_metrics: Optional[dict],
    fiscal_quarter: Optional[int],
) -> tuple[dict, bool]:
    raw_metrics = extract_metrics(xbrl, period_end, form_type)
    if not raw_metrics:
        return raw_metrics, False
    return decumulate_10q_metrics(
        current_metrics  = raw_metrics,
        prev_ytd_metrics = prev_ytd_metrics or {},
        fiscal_quarter   = fiscal_quarter or 0,
        form_type        = form_type,
    )


# ─── FIX-2: QUARTER-ALIGNED YoY ───────────────────────────────────────────────

def compute_quarter_aligned_yoy(
    current_metrics: dict,
    current_period: str,
    current_quarter: int,
    historical_records: list[dict],
    form_type: str,
) -> tuple[dict, bool]:
    if not current_period or not historical_records:
        return {}, False
    try:
        cur_dt = datetime.strptime(current_period[:10], "%Y-%m-%d")
    except ValueError:
        return {}, False

    target_dt   = cur_dt - timedelta(days=365)
    window_low  = target_dt - timedelta(days=30)
    window_high = target_dt + timedelta(days=30)

    prior_metrics = None
    for rec in historical_records:
        rec_form   = rec.get("meta", {}).get("form_type", "")
        rec_period = rec.get("meta", {}).get("period_of_report", "")
        rec_q      = rec.get("meta", {}).get("fiscal_quarter")
        if form_type.split("/")[0] not in rec_form:
            continue
        if rec_q is not None and rec_q != current_quarter:
            continue
        try:
            rec_dt = datetime.strptime(rec_period[:10], "%Y-%m-%d")
        except ValueError:
            continue
        if window_low <= rec_dt <= window_high:
            prior_metrics = rec.get("metrics", {})
            logger.debug(f"    Quarter-aligned YoY: {current_period} vs {rec_period}")
            break

    if prior_metrics is None:
        return {}, False
    return compute_changes(current_metrics, prior_metrics), True


# ─── FIX-3: SANITY VALIDATION ─────────────────────────────────────────────────

def sanity_check_metrics(
    metrics: dict,
    changes: dict,
    ticker: str,
    period: str,
    form_type: str,
    is_mega_cap: bool = True,
) -> dict:
    flags     = []
    action    = "pass"
    rev_yoy   = changes.get("revenue_yoy_pct")
    ni_yoy    = changes.get("net_income_yoy_pct")
    gm_d      = changes.get("gross_margin_pct_delta")
    fcf_yoy   = changes.get("fcf_yoy_pct")
    threshold = MEGA_CAP_SWING_THRESHOLD if is_mega_cap else 50.0

    if rev_yoy is not None and abs(rev_yoy) > threshold:
        flags.append(f"REVENUE_SWING: {rev_yoy:+.1f}% exceeds ±{threshold}% — possible YTD/discrete mismatch")
        action = "flag"
    if gm_d is not None and abs(gm_d) > 40:
        flags.append(f"MARGIN_DELTA_IMPOSSIBLE: Δ={gm_d:+.1f}pp — physically implausible")
        action = "block"
    if rev_yoy is not None and rev_yoy < -50:
        flags.append(f"REVENUE_COLLAPSE: {rev_yoy:+.1f}% — likely YTD vs discrete mismatch")
        action = "block"
    if rev_yoy is not None and rev_yoy > 100 and is_mega_cap:
        flags.append(f"HYPERGROWTH_ARTIFACT: {rev_yoy:+.1f}% — implausible for mega-cap")
        action = "block"
    if rev_yoy is not None and ni_yoy is not None and abs(ni_yoy - rev_yoy) > 80:
        flags.append(f"INCOME_REVENUE_DIVERGENCE: rev={rev_yoy:+.1f}% vs ni={ni_yoy:+.1f}%")
        if action == "pass":
            action = "flag"
    if fcf_yoy is not None and rev_yoy is not None and fcf_yoy > 30 and rev_yoy < -20:
        flags.append(f"FCF_REV_PARADOX: FCF +{fcf_yoy:.1f}% while revenue {rev_yoy:+.1f}%")
        if action == "pass":
            action = "flag"

    confidence = "high" if not flags else ("medium" if action == "flag" else "low")
    note       = "Data validated." if not flags else f"{len(flags)} rule(s) triggered. Action: {action.upper()}."
    return {"action": action, "flags": flags, "confidence": confidence, "note": note,
            "ticker": ticker, "period": period, "form_type": form_type}


# ─── FIX-4: PERIOD TYPE METADATA ──────────────────────────────────────────────

def enrich_period_type_metadata(
    meta: dict,
    fiscal_quarter: Optional[int],
    decumulation_applied: bool,
    quarter_aligned_yoy: bool,
    sanity_result: dict,
) -> dict:
    ft = meta.get("form_type", "")
    if ft in ("10-K", "10-K/A"):
        period_type = "full_year"
    elif fiscal_quarter == 1:
        period_type = "discrete_quarter"
    elif fiscal_quarter in (2, 3):
        period_type = "discrete_quarter" if decumulation_applied else "ytd_cumulative"
    elif fiscal_quarter == 4:
        period_type = "full_year"
    else:
        period_type = "unknown"

    meta["period_type"]          = period_type
    meta["fiscal_quarter"]       = fiscal_quarter
    meta["decumulation_applied"] = decumulation_applied
    meta["quarter_aligned_yoy"]  = quarter_aligned_yoy
    meta["sanity_action"]        = sanity_result.get("action", "unknown")
    meta["sanity_confidence"]    = sanity_result.get("confidence", "unknown")
    meta["sanity_flags_count"]   = len(sanity_result.get("flags", []))
    return meta


# ─── YoY CHANGES ──────────────────────────────────────────────────────────────

def compute_changes(current: dict, previous: dict) -> dict:
    PAIRS = [
        ("revenue",             "revenue_yoy_pct"),
        ("net_income",          "net_income_yoy_pct"),
        ("gross_profit",        "gross_profit_yoy_pct"),
        ("operating_income",    "operating_income_yoy_pct"),
        ("ebitda",              "ebitda_yoy_pct"),
        ("free_cash_flow",      "fcf_yoy_pct"),
        ("operating_cash_flow", "ocf_yoy_pct"),
        ("long_term_debt",      "debt_yoy_pct"),
        ("total_assets",        "assets_yoy_pct"),
        ("total_equity",        "equity_yoy_pct"),
        ("research_development","rd_yoy_pct"),
        ("eps_diluted",         "eps_yoy_pct"),
        ("capex",               "capex_yoy_pct"),
    ]
    c = {}
    for metric, label in PAIRS:
        cur  = current.get(metric)
        prev = previous.get(metric)
        if cur is not None and prev and prev != 0:
            c[label] = round(((cur - prev) / abs(prev)) * 100, 2)

    for margin in ("gross_margin_pct", "net_margin_pct", "operating_margin_pct",
                   "cogs_ratio_pct", "sga_ratio_pct", "rd_ratio_pct"):
        cur  = current.get(margin)
        prev = previous.get(margin)
        if cur is not None and prev is not None:
            c[f"{margin}_delta"] = round(cur - prev, 2)

    rev = c.get("revenue_yoy_pct")
    if rev is not None:
        c["trend_revenue"] = (
            "hypergrowth"        if rev > 50  else
            "strong_growth"      if rev > 20  else
            "moderate_growth"    if rev > 5   else
            "slowing_growth"     if rev > 0   else
            "mild_decline"       if rev > -10 else
            "significant_decline"
        )
    gm = c.get("gross_margin_pct_delta")
    if gm is not None:
        c["trend_margin"] = "expanding" if gm > 1 else "compressing" if gm < -1 else "stable"
    fcf = c.get("fcf_yoy_pct")
    if fcf is not None:
        c["trend_fcf"] = "strong_generation" if fcf > 20 else "improving" if fcf > 0 else "deteriorating"
    dbt = c.get("debt_yoy_pct")
    if dbt is not None:
        c["trend_leverage"] = (
            "rapidly_increasing"    if dbt > 30  else
            "moderately_increasing" if dbt > 10  else
            "stable"                if -10 <= dbt <= 10 else
            "decreasing"
        )
    return c


def compute_signals(m: dict, c: dict) -> dict:
    s  = {}
    nm = m.get("net_margin_pct", 0)
    s["profitability_tier"] = (
        "exceptional" if nm > 30 else "strong" if nm > 15 else
        "average"     if nm > 5  else "weak"   if nm > 0  else "loss_making"
    )
    ni  = m.get("net_income",    0) or 0
    fcf = m.get("free_cash_flow",0) or 0
    if ni > 0:
        r = fcf / ni
        s["cash_conversion"] = (
            "excellent" if r > 1.1 else "good" if r > 0.8 else
            "moderate"  if r > 0.5 else "poor"
        )
    dr = m.get("debt_to_equity")
    cr = m.get("current_ratio")
    if dr is not None:
        s["leverage_health"] = (
            "minimal_debt"      if dr < 0.3 else "conservative"      if dr < 1.0 else
            "moderate_leverage" if dr < 2.0 else "high_leverage"      if dr < 4.0 else "overleveraged"
        )
    if cr is not None:
        s["liquidity_health"] = (
            "very_strong" if cr > 3.0 else "strong" if cr > 2.0 else
            "adequate"    if cr > 1.0 else "tight"
        )
    rev_g = c.get("revenue_yoy_pct", 0) or 0
    op_m  = m.get("operating_margin_pct", 0) or 0
    r40   = rev_g + op_m
    s["rule_of_40_score"] = round(r40, 1)
    s["rule_of_40_pass"]  = r40 >= 40
    rev = m.get("revenue", 0) or 0
    rd  = m.get("research_development", 0) or 0
    if rev > 0 and rd > 0:
        rdi = rd / rev * 100
        s["rd_intensity_pct"]      = round(rdi, 2)
        s["innovation_investment"] = (
            "heavy" if rdi > 20 else "moderate" if rdi > 10 else
            "light" if rdi > 3  else "minimal"
        )
    edge_cases = []
    for name, archetype in EDGE_CASE_ARCHETYPES.items():
        try:
            if archetype["condition"](m, c):
                edge_cases.append(name)
        except Exception:
            pass
    if edge_cases:
        s["edge_cases_detected"] = edge_cases
    return s


# ─── MULTI-HORIZON BUILDER ────────────────────────────────────────────────────

def build_multi_horizon_block(company_records: list[dict], form_type: str) -> Optional[dict]:
    recs = sorted(
        [r for r in company_records
         if r["meta"].get("form_type") == form_type and r.get("metrics", {}).get("revenue")],
        key=lambda r: r["meta"].get("period_of_report", ""),
    )
    if len(recs) < 2:
        return None
    latest   = recs[-1]
    prev_rec = recs[-2]
    horizons = {}

    if "10-Q" in form_type:
        cur_rev  = latest["metrics"].get("revenue")
        prev_rev = prev_rec["metrics"].get("revenue")
        if cur_rev and prev_rev and prev_rev != 0:
            qoq = round(((cur_rev - prev_rev) / abs(prev_rev)) * 100, 2)
            horizons["qoq_revenue_pct"] = qoq
            horizons["qoq_label"] = "accelerating" if qoq > 5 else "stable" if qoq > -5 else "decelerating"

    cur_rev  = latest["metrics"].get("revenue")
    prev_rev = prev_rec["metrics"].get("revenue")
    if cur_rev and prev_rev and prev_rev != 0:
        horizons["yoy_revenue_pct"] = round(((cur_rev - prev_rev) / abs(prev_rev)) * 100, 2)

    if len(recs) >= 3:
        start_rec = recs[-3]
        start_rev = start_rec["metrics"].get("revenue")
        end_rev   = latest["metrics"].get("revenue")
        start_per = start_rec["meta"].get("period_of_report", "")
        end_per   = latest["meta"].get("period_of_report", "")
        if start_rev and end_rev and start_rev > 0:
            try:
                y1      = datetime.strptime(start_per[:10], "%Y-%m-%d")
                y2      = datetime.strptime(end_per[:10],   "%Y-%m-%d")
                n_years = max((y2 - y1).days / 365.25, 0.5)
                horizons["three_year_cagr_pct"]     = round(((end_rev / start_rev) ** (1 / n_years) - 1) * 100, 2)
                horizons["three_year_start_period"] = start_per
                horizons["three_year_end_period"]   = end_per
            except Exception:
                pass

    cur_gm = latest["metrics"].get("gross_margin_pct")
    old_gm = recs[0]["metrics"].get("gross_margin_pct")
    if cur_gm is not None and old_gm is not None:
        horizons["gross_margin_trajectory_pp"] = round(cur_gm - old_gm, 2)
        horizons["gross_margin_start"]         = old_gm
        horizons["gross_margin_end"]           = cur_gm

    revenues = [r["metrics"].get("revenue") for r in recs if r["metrics"].get("revenue")]
    if len(revenues) >= 4:
        mean = sum(revenues) / len(revenues)
        std  = (sum((rv - mean) ** 2 for rv in revenues) / len(revenues)) ** 0.5
        cv   = round(std / mean * 100, 1) if mean > 0 else 0
        horizons["revenue_volatility_cv_pct"] = cv
        horizons["cyclicality"] = "high" if cv > 20 else "moderate" if cv > 10 else "low"

    return horizons if horizons else None


# ─── NEGATIVE EXAMPLES ────────────────────────────────────────────────────────

def build_negative_examples(metrics: dict, changes: dict, text_sections: dict) -> list[dict]:
    mda_text     = text_sections.get("mda", "") or ""
    mda_has_text = len(mda_text) > MIN_SECTION_CHARS
    negatives    = []
    for tmpl in NEGATIVE_REASONING_TEMPLATES:
        try:
            ctx    = {**metrics, **changes, "mda_has_text": mda_has_text}
            result = eval(tmpl["trigger"], {"__builtins__": {}}, ctx)
            if result:
                fmt_ctx = {k: (v if v is not None else 0) for k, v in ctx.items()}
                try:
                    good = tmpl["correct_conclusion"].format(**fmt_ctx)
                except (KeyError, ValueError):
                    good = tmpl["correct_conclusion"]
                negatives.append({
                    "bad_conclusion":     tmpl["bad_conclusion"],
                    "correct_conclusion": good,
                    "label":              tmpl["label"],
                })
        except Exception:
            continue
    return negatives


# ─── PROMPT BUILDERS ──────────────────────────────────────────────────────────

def build_single_prompt(
    record: dict,
    prior_record: Optional[dict] = None,
    multi_horizon: Optional[dict] = None,
) -> str:
    m, c, s   = record.get("metrics", {}), record.get("changes", {}), record.get("signals", {})
    txt, meta = record.get("text_sections", {}), record.get("meta", {})
    mda_text  = (txt.get("mda", "") or "")
    risk_text = (txt.get("risk_factors", "") or "")[:500]
    has_mda   = len(mda_text) > MIN_SECTION_CHARS

    causality_instr = CausalGuard.build_causality_instruction(m, c, mda_text)
    grounded_ctx    = MDAGrounder.build_grounded_context(m, c, mda_text)

    conf = EvidenceConfidence.score(m, c, txt)
    conf_block = (
        f"EVIDENCE CONFIDENCE: {conf['level']} "
        f"(metric_cov={conf['metric_coverage']:.0%}, "
        f"change_cov={conf['change_coverage']:.0%}, "
        f"MD&A={'YES' if conf['has_mda'] else 'NO'})"
    )
    if conf["qualifier"]:
        conf_block += f"\n{conf['qualifier']}"

    period_type_block = (
        f"DATA PROVENANCE: period_type={meta.get('period_type','unknown')} | "
        f"fiscal_Q={meta.get('fiscal_quarter')} | "
        f"de-cumulated={'YES' if meta.get('decumulation_applied') else 'NO'} | "
        f"sanity={meta.get('sanity_action','pass')}"
    )

    # FIX-D: Abnormal return context in prompt
    ar = meta.get("abnormal_return", {})
    abnormal_block = ""
    if ar:
        abnormal_block = (
            f"\nMARKET REACTION ({ar.get('window_days',5)}-day post-filing): "
            f"stock={ar.get('stock_return_pct','N/A'):+.2f}% | "
            f"SPY={ar.get('spy_return_pct','N/A'):+.2f}% | "
            f"abnormal={ar.get('abnormal_return_pct','N/A'):+.2f}% | "
            f"signal={ar.get('market_reaction','N/A')}"
        )

    edge_cases = s.get("edge_cases_detected", [])
    edge_instr = ""
    if edge_cases:
        focuses    = [EDGE_CASE_ARCHETYPES[ec]["reasoning_focus"]
                      for ec in edge_cases if ec in EDGE_CASE_ARCHETYPES]
        edge_instr = "\nEDGE CASE ALERT:\n" + "\n".join(f"• {f}" for f in focuses)

    ordering     = StructuralRandomizer.get_ordering(seed=record.get("id", ""))
    schema_block = StructuralRandomizer.build_output_schema_instruction(ordering)

    prior_block = ""
    if prior_record and prior_record.get("metrics"):
        pm = prior_record["metrics"]
        pp = prior_record["meta"].get("period_of_report", "prior")
        prior_block = (
            f"\n── PRIOR PERIOD ({pp}) ──\n"
            f"Rev: {_usd(pm.get('revenue'))} | GM: {_pct_plain(pm.get('gross_margin_pct'))} | "
            f"NM: {_pct_plain(pm.get('net_margin_pct'))} | FCF: {_usd(pm.get('free_cash_flow'))} | "
            f"EPS: {pm.get('eps_diluted','N/A')} | D/E: {pm.get('debt_to_equity','N/A')}\n"
        )

    horizon_block = ""
    if multi_horizon:
        parts = []
        if "qoq_revenue_pct"     in multi_horizon: parts.append(f"QoQ Rev: {_pct(multi_horizon['qoq_revenue_pct'])} ({multi_horizon.get('qoq_label','')})")
        if "yoy_revenue_pct"     in multi_horizon: parts.append(f"YoY Rev: {_pct(multi_horizon['yoy_revenue_pct'])}")
        if "three_year_cagr_pct" in multi_horizon: parts.append(f"3yr CAGR: {_pct(multi_horizon['three_year_cagr_pct'])}")
        if "gross_margin_trajectory_pp" in multi_horizon:
            parts.append(f"GM: {multi_horizon.get('gross_margin_start','?')}% → {multi_horizon.get('gross_margin_end','?')}%")
        if "cyclicality" in multi_horizon:
            parts.append(f"Cyclicality: {multi_horizon['cyclicality']} (CV={multi_horizon.get('revenue_volatility_cv_pct','?')}%)")
        if parts:
            horizon_block = "\n── MULTI-HORIZON ──\n" + " | ".join(parts) + "\n"

    return f"""You are a senior equity research analyst. Produce a precise, evidence-grounded financial assessment.

COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | PERIOD: {meta.get('period_of_report')} | FORM: {meta.get('form_type')}
{conf_block}
{period_type_block}{abnormal_block}{prior_block}{horizon_block}
── CURRENT PERIOD ({meta.get('period_of_report')}) ──────────────────────────
Revenue:          {_usd(m.get('revenue'))}  YoY: {_pct(c.get('revenue_yoy_pct'))}
Gross Profit:     {_usd(m.get('gross_profit'))}  Margin: {_pct_plain(m.get('gross_margin_pct'))}  Δ: {_pct(c.get('gross_margin_pct_delta'),'pp')}
COGS ratio:       {_pct_plain(m.get('cogs_ratio_pct'))}  Δ: {_pct(c.get('cogs_ratio_pct_delta'),'pp')}
SG&A ratio:       {_pct_plain(m.get('sga_ratio_pct'))}  Δ: {_pct(c.get('sga_ratio_pct_delta'),'pp')}
Operating Income: {_usd(m.get('operating_income'))}  Margin: {_pct_plain(m.get('operating_margin_pct'))}
EBITDA:           {_usd(m.get('ebitda'))}  YoY: {_pct(c.get('ebitda_yoy_pct'))}
Net Income:       {_usd(m.get('net_income'))}  Margin: {_pct_plain(m.get('net_margin_pct'))}  YoY: {_pct(c.get('net_income_yoy_pct'))}
R&D:              {_usd(m.get('research_development'))}  ({_pct_plain(s.get('rd_intensity_pct'))} of rev)  YoY: {_pct(c.get('rd_yoy_pct'))}
EPS (diluted):    {m.get('eps_diluted','N/A')}  YoY: {_pct(c.get('eps_yoy_pct'))}

FCF:              {_usd(m.get('free_cash_flow'))}  YoY: {_pct(c.get('fcf_yoy_pct'))}  FCF/NI: {m.get('fcf_conversion_ratio','N/A')}
OCF:              {_usd(m.get('operating_cash_flow'))}  YoY: {_pct(c.get('ocf_yoy_pct'))}
CapEx:            {_usd(m.get('capex'))}  YoY: {_pct(c.get('capex_yoy_pct'))}  Buybacks: {_usd(m.get('share_repurchases'))}  SBC: {_usd(m.get('stock_based_compensation'))}

Cash:             {_usd(m.get('cash_and_equivalents'))} | Net Debt: {_usd(m.get('net_debt'))}
LT Debt:          {_usd(m.get('long_term_debt'))}  YoY: {_pct(c.get('debt_yoy_pct'))} | D/E: {m.get('debt_to_equity','N/A')} | CR: {m.get('current_ratio','N/A')}
Total Assets:     {_usd(m.get('total_assets'))} | ROA: {_pct_plain(m.get('return_on_assets_pct'))} | ROE: {_pct_plain(m.get('return_on_equity_pct'))}

Profitability: {s.get('profitability_tier','N/A')} | Cash conv: {s.get('cash_conversion','N/A')}
Leverage: {s.get('leverage_health','N/A')} | Liquidity: {s.get('liquidity_health','N/A')}
Rule-of-40: {s.get('rule_of_40_score','N/A')} ({'PASS' if s.get('rule_of_40_pass') else 'FAIL'})
Trends: rev={c.get('trend_revenue','N/A')} | margin={c.get('trend_margin','N/A')} | fcf={c.get('trend_fcf','N/A')}
{edge_instr}
── FILING TEXT ──────────────────────────────────────────────────
MD&A ({len(mda_text)} chars): {mda_text[:900] or '[not available]'}
Risk Factors: {risk_text or '[not available]'}

{grounded_ctx}

── ANALYSIS INSTRUCTIONS ────────────────────────────────────────
STYLE: Every claim needs magnitude+direction+timeframe. No filler words. 120-180 words.
{causality_instr}

Return ONLY valid JSON:
{schema_block}

investment_signal: STRONG_BUY | BUY | HOLD | REDUCE | SELL
key_risks: 3 items with specific metric references.
signal_rationale: 1 sentence with ≥2 specific metrics."""


def build_negative_example_prompt(record: dict, negatives: list[dict]) -> str:
    m, c  = record.get("metrics", {}), record.get("changes", {})
    meta  = record.get("meta", {})
    neg_block = "\n".join(
        f"  SCENARIO {i+1}: {neg['label'].replace('_',' ')}.\n"
        f"    ❌ INVALID: \"{neg['bad_conclusion']}\"\n"
        f"    ✓ CORRECT:  \"{neg['correct_conclusion']}\""
        for i, neg in enumerate(negatives)
    )
    return f"""You are a financial reasoning teacher. Explain WHY each invalid conclusion is logically unsupported.

COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | PERIOD: {meta.get('period_of_report')}
Revenue YoY: {_pct(c.get('revenue_yoy_pct'))} | GM Δ: {_pct(c.get('gross_margin_pct_delta'),'pp')} | Net Margin: {_pct_plain(m.get('net_margin_pct'))}
FCF YoY: {_pct(c.get('fcf_yoy_pct'))} | D/E: {m.get('debt_to_equity','N/A')} | CR: {m.get('current_ratio','N/A')}

{neg_block}

Return ONLY valid JSON:
{{
  "negative_examples": [
    {{
      "scenario_label": "<label>",
      "why_invalid": "<1-2 sentences>",
      "correct_reasoning": "<1-2 sentences with specific numbers>",
      "reasoning_boundary": "<what evidence WOULD support the invalid claim>"
    }}
  ],
  "epistemic_principle": "<1 sentence general principle>"
}}"""


def build_multiyear_prompt(
    ticker: str, company: str, form_type: str,
    timeline: list[dict], multi_horizon: Optional[dict] = None,
) -> str:
    lines = []
    for entry in timeline:
        p, m, c = entry["period"], entry["metrics"], entry["changes"]
        rev_yoy = f" ({_pct(c.get('revenue_yoy_pct'))} YoY)" if c.get("revenue_yoy_pct") is not None else ""
        lines.append(
            f"{p}: Rev={_usd(m.get('revenue'))}{rev_yoy} | "
            f"GM={_pct_plain(m.get('gross_margin_pct'))} | NM={_pct_plain(m.get('net_margin_pct'))} | "
            f"FCF={_usd(m.get('free_cash_flow'))} | {c.get('trend_revenue','N/A')}"
        )
    horizon_block = ""
    if multi_horizon:
        parts = []
        if "three_year_cagr_pct" in multi_horizon: parts.append(f"3yr CAGR: {_pct(multi_horizon['three_year_cagr_pct'])}")
        if "gross_margin_trajectory_pp" in multi_horizon:
            parts.append(f"GM: {multi_horizon.get('gross_margin_start','?')}% → {multi_horizon.get('gross_margin_end','?')}%")
        if "cyclicality" in multi_horizon: parts.append(f"Cyclicality: {multi_horizon['cyclicality']}")
        if parts: horizon_block = "\nPRE-COMPUTED: " + " | ".join(parts) + "\n"

    return f"""You are a senior equity research analyst. Analyze {len(timeline)}-period evolution of {company} ({ticker}) — {form_type}.
Period: {timeline[0]['period']} → {timeline[-1]['period']}
{horizon_block}
TIMELINE (oldest → newest):
{chr(10).join(lines)}

CAUSALITY: Base ALL analysis on timeline numbers. No product names. Quantified claims only.

Return ONLY valid JSON:
{{
  "trend_summary": "<CAGR + margin trajectory — 2 sentences max>",
  "key_inflection_points": ["<period: metric Δ with numbers>", "<period: next>"],
  "revenue_cagr": "<formula + result>",
  "margin_evolution": "<exact start%→end%, Δpp>",
  "cash_generation_quality": "<FCF trend, CapEx%, FCF/NI>",
  "risk_evolution": "<D/E start→end, liquidity>",
  "investment_thesis": "<bull: metric | bear: metric risk>",
  "overall_signal": "IMPROVING | STABLE | DETERIORATING | MIXED"
}}"""




def generate_analysis(prompt: str, llm) -> str:
    """
    FIX-B: LLM call with adaptive rate limit management.
    Uses RateLimitManager for circuit breaking + jittered backoff.
    """
    global _rate_limiter

    if _rate_limiter.is_circuit_open():
        _rate_limiter.wait_circuit()

    for attempt in range(MAX_RETRIES):
        try:
            text = llm.invoke(prompt).content.strip()
            m    = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, re.DOTALL)
            if m:
                text = m.group(1)
            json.loads(text)
            text = normalize_financial_terms(text)
            _rate_limiter.record_success()
            return text
        except json.JSONDecodeError:
            _rate_limiter.record_success()
            return text if text else ""
        except Exception as e:
            err = str(e)
            if "429" in err or "rate_limit" in err.lower():
                wait = _rate_limiter.parse_wait_from_error(err)
                _rate_limiter.record_429(wait)

                if wait > LLM_429_MAX_WAIT:
                    logger.warning(f"  429: wait {wait}s > max {LLM_429_MAX_WAIT}s — skipping")
                    _rate_limiter.record_skip()
                    return ""

                jittered = _rate_limiter.jitter(wait)
                logger.warning(
                    f"  429 → sleep {jittered:.1f}s (attempt {attempt+1}/{MAX_RETRIES}) | "
                    f"consecutive={_rate_limiter.consecutive_failures}"
                )
                time.sleep(jittered)

                if _rate_limiter.is_circuit_open():
                    _rate_limiter.wait_circuit()
                continue
            logger.warning(f"  LLM error: {e}")
            return ""
    _rate_limiter.record_skip()
    return ""


# ─── REASONING TYPE CLASSIFIER ────────────────────────────────────────────────

def classify_reasoning_types(
    metrics: dict, changes: dict, extra_flags: Optional[dict] = None
) -> list[str]:
    types = [k for k, fn in REASONING_TYPES.items() if fn(metrics, changes)]
    if extra_flags:
        for flag, active in extra_flags.items():
            if active and flag in REASONING_TYPES and flag not in types:
                types.append(flag)
    return types


# ─── PAIR BUILDERS ────────────────────────────────────────────────────────────

def build_single_pair(
    record: dict,
    analysis: str,
    prior_record: Optional[dict] = None,
    multi_horizon: Optional[dict] = None,
    quality_result: Optional[dict] = None,
) -> dict:
    m, c, s   = record.get("metrics", {}), record.get("changes", {}), record.get("signals", {})
    txt, meta = record.get("text_sections", {}), record.get("meta", {})
    has_text  = any(v for v in txt.values() if v and len(v) >= MIN_SECTION_CHARS)
    risks     = [kw for kw in RISK_KW
                 if kw in ((txt.get("risk_factors","") or "") + (txt.get("mda","") or "")).lower()][:6]

    conf            = EvidenceConfidence.score(m, c, txt)
    reasoning_types = classify_reasoning_types(m, c)
    edge_cases      = s.get("edge_cases_detected", [])
    if edge_cases:
        reasoning_types.append("edge_case")

    ar = meta.get("abnormal_return", {})
    if ar:
        reasoning_types.append("market_impact")

    prior_section = ""
    if prior_record and prior_record.get("metrics"):
        pm = prior_record["metrics"]
        pp = prior_record["meta"].get("period_of_report", "prior")
        prior_section = (
            f"\nPRIOR ({pp}): Rev={_usd(pm.get('revenue'))} | "
            f"GM={_pct_plain(pm.get('gross_margin_pct'))} | NM={_pct_plain(pm.get('net_margin_pct'))} | "
            f"FCF={_usd(pm.get('free_cash_flow'))} | EPS={pm.get('eps_diluted','N/A')}"
        )

    horizon_section = ""
    if multi_horizon:
        h_parts = []
        if "qoq_revenue_pct"     in multi_horizon: h_parts.append(f"QoQ Rev: {_pct(multi_horizon['qoq_revenue_pct'])}")
        if "three_year_cagr_pct" in multi_horizon: h_parts.append(f"3yr CAGR: {_pct(multi_horizon['three_year_cagr_pct'])}")
        if "cyclicality"         in multi_horizon: h_parts.append(f"Cyclicality: {multi_horizon['cyclicality']}")
        if h_parts:
            horizon_section = "\nHORIZONS: " + " | ".join(h_parts)

    abnormal_section = ""
    if ar:
        abnormal_section = (
            f"\nMARKET REACTION: abnormal={ar.get('abnormal_return_pct','N/A'):+.2f}% "
            f"over {ar.get('window_days',5)}d post-filing | signal={ar.get('market_reaction','N/A')}"
        )

    inp = (
        f"COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | "
        f"FORM: {meta.get('form_type')} | PERIOD: {meta.get('period_of_report')} | "
        f"period_type={meta.get('period_type','unknown')} | "
        f"fiscal_Q={meta.get('fiscal_quarter','N/A')} | "
        f"de-cumulated={'YES' if meta.get('decumulation_applied') else 'NO'}"
        f"{prior_section}{horizon_section}{abnormal_section}\n\n"
        f"── INCOME ──\n"
        f"Revenue: {_usd(m.get('revenue'))} (YoY {_pct(c.get('revenue_yoy_pct'))})\n"
        f"Gross Margin: {_pct_plain(m.get('gross_margin_pct'))} Δ{_pct(c.get('gross_margin_pct_delta'),'pp')} | "
        f"COGS: {_pct_plain(m.get('cogs_ratio_pct'))}\n"
        f"Net Income: {_usd(m.get('net_income'))} | Margin: {_pct_plain(m.get('net_margin_pct'))} | "
        f"YoY: {_pct(c.get('net_income_yoy_pct'))}\n"
        f"EPS: {m.get('eps_diluted','N/A')} (YoY {_pct(c.get('eps_yoy_pct'))})\n\n"
        f"── CASH FLOW ──\n"
        f"FCF: {_usd(m.get('free_cash_flow'))} (YoY {_pct(c.get('fcf_yoy_pct'))}) | FCF/NI: {m.get('fcf_conversion_ratio','N/A')}\n"
        f"CapEx: {_usd(m.get('capex'))} | Buybacks: {_usd(m.get('share_repurchases'))}\n\n"
        f"── BALANCE SHEET ──\n"
        f"Net Debt: {_usd(m.get('net_debt'))} | D/E: {m.get('debt_to_equity','N/A')} | "
        f"CR: {m.get('current_ratio','N/A')} | ROE: {_pct_plain(m.get('return_on_equity_pct'))}\n\n"
        f"── SIGNALS ──\n"
        f"{s.get('profitability_tier','N/A')} | "
        f"Rule-of-40: {s.get('rule_of_40_score','N/A')} ({'PASS' if s.get('rule_of_40_pass') else 'FAIL'}) | "
        f"Trend: {c.get('trend_revenue','N/A')} | Margin: {c.get('trend_margin','N/A')}"
    )
    if has_text:
        inp += f"\n\n── MD&A ──\n{(txt.get('mda','') or '')[:500]}"
        inp += f"\n\n── RISKS ──\n{(txt.get('risk_factors','') or '')[:300]}"
    else:
        inp += "\n\n[MD&A not available — quantitative analysis only]"

    if edge_cases:
        inp += f"\n\n── EDGE CASES ──\n{', '.join(edge_cases)}"

    return {
        "instruction": "You are a senior financial analyst. Analyze the SEC filing data and return a structured JSON assessment with evidence-grounded causal reasoning.",
        "input":       inp,
        "output":      analysis,
        "record_type": "single_period",
        "metadata": {
            "ticker":               meta.get("ticker"),
            "company":              meta.get("company_name"),
            "period":               meta.get("period_of_report"),
            "form_type":            meta.get("form_type"),
            "output_schema":        "v15_full",
            "analysis_mode":        "full" if has_text else "quantitative_only",
            "evidence_confidence":  conf["level"],
            "period_type":          meta.get("period_type", "unknown"),
            "fiscal_quarter":       meta.get("fiscal_quarter"),
            "decumulation_applied": meta.get("decumulation_applied", False),
            "quarter_aligned_yoy":  meta.get("quarter_aligned_yoy", False),
            "sanity_action":        meta.get("sanity_action", "unknown"),
            "profitability":        s.get("profitability_tier"),
            "trend_revenue":        c.get("trend_revenue"),
            "trend_margin":         c.get("trend_margin"),
            "trend_fcf":            c.get("trend_fcf"),
            "edge_cases":           edge_cases,
            "reasoning_types":      reasoning_types,
            "risk_keywords":        risks,
            "rule_of_40_pass":      s.get("rule_of_40_pass"),
            "metrics_count":        len(m),
            "text_sections_filled": sum(1 for v in txt.values() if v and len(v) >= MIN_SECTION_CHARS),
            "has_mda":              has_text,
            "quality_score":        quality_result.get("quality_score") if quality_result else None,
            "quality_tier":         quality_result.get("tier") if quality_result else None,
            "hallucination_flag":   quality_result.get("hallucination", {}).get("flag") if quality_result else None,
            "abnormal_return_pct":  ar.get("abnormal_return_pct") if ar else None,
            "market_reaction":      ar.get("market_reaction") if ar else None,
        },
    }


def build_negative_pair(record: dict, neg_analysis: str, negatives: list[dict]) -> dict:
    m, c, meta = record.get("metrics", {}), record.get("changes", {}), record.get("meta", {})
    inp = (
        f"COMPANY: {meta.get('company_name')} ({meta.get('ticker')}) | "
        f"FORM: {meta.get('form_type')} | PERIOD: {meta.get('period_of_report')}\n"
        f"Revenue YoY: {_pct(c.get('revenue_yoy_pct'))} | "
        f"GM Δ: {_pct(c.get('gross_margin_pct_delta'),'pp')} | "
        f"Net Margin: {_pct_plain(m.get('net_margin_pct'))}\n"
        f"FCF YoY: {_pct(c.get('fcf_yoy_pct'))} | D/E: {m.get('debt_to_equity','N/A')} | CR: {m.get('current_ratio','N/A')}\n\n"
        "Identify what conclusions CANNOT be drawn from this data:\n"
        + "\n".join(f"Scenario {i+1}: {n['label']}" for i, n in enumerate(negatives))
    )
    return {
        "instruction": "You are a financial reasoning teacher. Identify logically unsupported conclusions and explain epistemic boundaries.",
        "input":       inp,
        "output":      neg_analysis,
        "record_type": "negative_reasoning",
        "metadata": {
            "ticker":          meta.get("ticker"),
            "company":         meta.get("company_name"),
            "period":          meta.get("period_of_report"),
            "form_type":       meta.get("form_type"),
            "output_schema":   "v15_negative",
            "reasoning_types": ["negative_inference"],
            "scenarios":       [n["label"] for n in negatives],
        },
    }


def build_multiyear_pair(
    ticker: str, company: str, form_type: str,
    timeline: list[dict], analysis: str,
    multi_horizon: Optional[dict] = None,
) -> dict:
    lines = [
        f"{e['period']}: Rev={_usd(e['metrics'].get('revenue'))} | "
        f"GM={_pct_plain(e['metrics'].get('gross_margin_pct'))} | "
        f"NM={_pct_plain(e['metrics'].get('net_margin_pct'))} | "
        f"FCF={_usd(e['metrics'].get('free_cash_flow'))}"
        for e in timeline
    ]
    horizon_line = ""
    if multi_horizon and "three_year_cagr_pct" in multi_horizon:
        horizon_line = f"\nCAGR: {_pct(multi_horizon['three_year_cagr_pct'])} | Cyclicality: {multi_horizon.get('cyclicality','N/A')}\n"
    inp = (
        f"Multi-year evolution of {company} ({ticker}) — {form_type} | "
        f"{timeline[0]['period']} → {timeline[-1]['period']}{horizon_line}\n"
        + "\n".join(lines)
    )
    return {
        "instruction": "You are a senior financial analyst. Analyze this multi-year financial evolution with evidence-grounded causality.",
        "input":       inp,
        "output":      analysis,
        "record_type": "multi_year",
        "metadata": {
            "ticker":          ticker,
            "company":         company,
            "form_type":       form_type,
            "output_schema":   "v15_multiyear",
            "period_start":    timeline[0]["period"],
            "period_end":      timeline[-1]["period"],
            "n_periods":       len(timeline),
            "reasoning_types": ["multi_horizon", "trend_analysis", "margin_analysis"],
        },
    }


# ─── MULTI-YEAR RECORDS ────────────────────────────────────────────────────────

def build_multiyear_records(
    company_records: list[dict],
    form_type: str,
    ticker: str,
    company_name: str,
    llm,
    multi_horizon: Optional[dict] = None,
) -> list[dict]:
    recs = sorted(
        [r for r in company_records
         if r["meta"].get("form_type") == form_type and len(r.get("metrics", {})) >= 5],
        key=lambda r: r["meta"].get("period_of_report", ""),
    )
    if len(recs) < 3:
        return []

    def _tl(subset):
        return [{"period": r["meta"]["period_of_report"],
                 "metrics": r["metrics"], "changes": r["changes"], "signals": r["signals"]}
                for r in subset]

    output = []

    if len(recs) >= 4:
        tl       = _tl(recs)
        analysis = generate_analysis(build_multiyear_prompt(ticker, company_name, form_type, tl, multi_horizon), llm)
        if analysis:
            rec_id = hashlib.md5(f"{ticker}_{form_type}_multiyear_{len(recs)}".encode()).hexdigest()[:12]
            output.append({
                "id":   rec_id,
                "meta": {"ticker": ticker, "company_name": company_name, "form_type": form_type,
                         "record_type": "multi_year",
                         "period_start": recs[0]["meta"]["period_of_report"],
                         "period_end":   recs[-1]["meta"]["period_of_report"],
                         "n_periods":    len(recs)},
                "metrics": {}, "changes": {}, "signals": {}, "text_sections": {},
                "instruction_pair": build_multiyear_pair(ticker, company_name, form_type, tl, analysis, multi_horizon),
            })
            logger.info(f"    ✓ Multi-year ({len(recs)} periods): {len(analysis)} chars")

    for i in range(len(recs) - 2):
        subset = recs[i:i+3]
        tl     = _tl(subset)
        sp, ep = subset[0]["meta"]["period_of_report"], subset[-1]["meta"]["period_of_report"]
        analysis = generate_analysis(build_multiyear_prompt(ticker, company_name, form_type, tl), llm)
        if analysis:
            rec_id = hashlib.md5(f"{ticker}_{form_type}_3yr_{sp}_{ep}".encode()).hexdigest()[:12]
            output.append({
                "id":   rec_id,
                "meta": {"ticker": ticker, "company_name": company_name, "form_type": form_type,
                         "record_type": "multi_year_3yr", "period_start": sp, "period_end": ep, "n_periods": 3},
                "metrics": {}, "changes": {}, "signals": {}, "text_sections": {},
                "instruction_pair": build_multiyear_pair(ticker, company_name, form_type, tl, analysis),
            })
            logger.info(f"    ✓ 3yr {sp}→{ep}: {len(analysis)} chars")

    return output


# ══════════════════════════════════════════════════════════════════════════════
# MAIN PIPELINE — v15
# ══════════════════════════════════════════════════════════════════════════════

def process_company(
    company: dict,
    sess: EDGARSession,
    llm,
    forms: tuple        = ("10-K", "10-Q", "8-K"),
    max_per_form: int   = 8,
    done_accessions: set = None,
    quality_log          = None,
    sanity_log           = None,
    gold_out             = None,
    silver_out           = None,
) -> list[dict]:
    ticker = company["ticker"]
    logger.info(f"══ {ticker} ({company['name']}) ══")
    done_accessions = done_accessions or set()

    cik = ticker_to_cik(ticker, sess)
    if not cik:
        return []

    xbrl = load_xbrl_facts(cik, sess)

    prev_ytd_metrics: dict[str, dict] = {}
    prev_records:     dict[str, dict] = {}
    all_records:      list[dict]      = []
    all_historical:   list[dict]      = []

    for form_type in forms:
        filings = get_filings(cik, form_type, sess, max_per_form)

        for filing in filings:
            accno  = filing["accession_no"]
            period = filing["period_of_report"]
            prim   = filing["primary_document"]
            filed  = filing["filed_date"]

            if accno in done_accessions:
                logger.debug(f"  Skip (done): {accno}")
                continue

            logger.info(f"  ▶ {form_type} | {period} | {accno}")
            rec_id   = hashlib.md5(f"{ticker}_{form_type}_{period}_{accno}".encode()).hexdigest()[:12]
            base_meta = {
                "ticker":           ticker,
                "company_name":     company["name"],
                "cik":              cik,
                "form_type":        form_type,
                "filed_date":       filed,
                "period_of_report": period,
                "fiscal_year":      int(period[:4]) if period else None,
                "accession_no":     accno,
            }

            # FIX-D: Fetch abnormal return
            ar = fetch_abnormal_return(ticker, filed)
            if ar:
                base_meta["abnormal_return"] = ar
                logger.info(f"    Abnormal return: {ar['abnormal_return_pct']:+.2f}% ({ar['market_reaction']})")

            # ── 8-K ──────────────────────────────────────────────────────────
            if "8-K" in form_type:
                event_text = get_8k_content(cik, accno, prim, sess)
                record     = {
                    "id": rec_id, "meta": base_meta,
                    "metrics": {}, "changes": {}, "signals": {},
                    "text_sections": {"event_text": event_text},
                }
                if is_material_8k(event_text):
                    analysis = generate_analysis(build_single_prompt(record), llm)
                    if analysis:
                        has_mda = len(event_text) > MIN_SECTION_CHARS
                        qr      = QualityValidator.validate(analysis, {}, {}, has_mda)
                        record["instruction_pair"] = build_single_pair(record, analysis, quality_result=qr)
                        _write_tiered(record, qr, quality_log, gold_out, silver_out)
                        logger.info(f"    ✓ 8-K [Q={qr['quality_score']}|tier={qr['tier']}]: {len(analysis)} chars")
                else:
                    logger.debug("    8-K not material")
                all_records.append(record)
                continue

            # ── 10-K / 10-Q ──────────────────────────────────────────────────
            fiscal_quarter = get_fiscal_quarter(period, form_type)
            prev_key       = f"{ticker}_{form_type}"
            prev_ytd       = prev_ytd_metrics.get(prev_key)

            metrics, decum_applied = extract_metrics_with_decumulation(
                xbrl, period, form_type, prev_ytd, fiscal_quarter
            ) if xbrl else ({}, False)

            if xbrl and metrics:
                prev_ytd_metrics[prev_key] = extract_metrics(xbrl, period, form_type)

            # FIX-A: Multi-strategy text extraction
            text_sections = scrape_text_sections(cik, accno, prim, form_type, sess)
            logger.info(f"    Metrics: {len(metrics)} | decum: {decum_applied}")

            # FIX-2: Quarter-aligned YoY
            prior_rec = prev_records.get(prev_key)
            qa_changes, quarter_aligned = compute_quarter_aligned_yoy(
                metrics, period, fiscal_quarter or 0, all_historical, form_type)

            if not qa_changes and prior_rec:
                qa_changes      = compute_changes(metrics, prior_rec.get("metrics", {}))
                quarter_aligned = False
            changes = qa_changes

            # FIX-3: Sanity check
            sanity_result = sanity_check_metrics(
                metrics, changes, ticker, period, form_type, is_mega_cap=True)

            if sanity_log and sanity_result["flags"]:
                sanity_log.write(json.dumps(sanity_result, ensure_ascii=False) + "\n")
                sanity_log.flush()

            if sanity_result["action"] == "block":
                logger.warning(f"    ✗ SANITY BLOCK: {sanity_result['note']}")
                base_meta["sanity_blocked"] = True

            # FIX-4: Period type metadata
            base_meta = enrich_period_type_metadata(
                base_meta, fiscal_quarter, decum_applied, quarter_aligned, sanity_result)

            signals = compute_signals(metrics, changes)
            record  = {
                "id": rec_id, "meta": base_meta,
                "metrics": metrics, "changes": changes,
                "signals": signals, "text_sections": text_sections,
            }

            if len(metrics) >= 5 and sanity_result["action"] != "block":
                analysis = generate_analysis(
                    build_single_prompt(record, prior_record=prior_rec), llm)

                if analysis:
                    has_mda = any(v for v in text_sections.values() if v and len(v) >= MIN_SECTION_CHARS)
                    qr      = QualityValidator.validate(analysis, metrics, changes, has_mda)

                    if quality_log:
                        quality_log.write(json.dumps({
                            "ticker": ticker, "period": period, "accno": accno,
                            "form": form_type, "decum": decum_applied,
                            "q_aligned": quarter_aligned, "sanity": sanity_result["action"],
                            "has_mda": has_mda, **qr,
                        }, ensure_ascii=False) + "\n")
                        quality_log.flush()

                    record["instruction_pair"] = build_single_pair(
                        record, analysis, prior_record=prior_rec, quality_result=qr)
                    _write_tiered(record, qr, None, gold_out, silver_out)

                    mode = "FULL" if has_mda else "QUANT"
                    logger.info(
                        f"    ✓ [{mode}|Q={qr['quality_score']}|tier={qr['tier']}|"
                        f"decum={decum_applied}|qa={quarter_aligned}]: {len(analysis)} chars"
                    )

                    # Negative examples
                    negatives = build_negative_examples(metrics, changes, text_sections)
                    if negatives:
                        neg_analysis = generate_analysis(
                            build_negative_example_prompt(record, negatives), llm)
                        if neg_analysis:
                            neg_record = {
                                "id": rec_id + "_neg", "meta": base_meta,
                                "metrics": metrics, "changes": changes,
                                "signals": signals, "text_sections": text_sections,
                                "instruction_pair": build_negative_pair(record, neg_analysis, negatives),
                            }
                            all_records.append(neg_record)
                            logger.info(f"    ✓ Negative examples: {len(negatives)} scenarios")
                else:
                    record["_needs_llm"] = True
                    logger.warning("    ✗ LLM pending")
            elif sanity_result["action"] == "block":
                logger.warning("    ✗ LLM skipped — sanity BLOCK")
            else:
                logger.warning(f"    ✗ LLM skipped: {len(metrics)} metrics")

            prev_records[prev_key] = record
            all_historical.append(record)
            all_records.append(record)

    # Multi-year
    for form_type in ("10-K", "10-Q"):
        try:
            mh        = build_multi_horizon_block(all_records, form_type)
            multiyear = build_multiyear_records(
                all_records, form_type, ticker, company["name"], llm, multi_horizon=mh)
            all_records.extend(multiyear)
            if multiyear:
                logger.success(f"  Multi-year {form_type}: +{len(multiyear)} records")
        except Exception as e:
            logger.warning(f"  Multi-year {form_type} failed: {e}")

    logger.success(f"  {ticker}: {len(all_records)} total records | {_rate_limiter.summary()}")
    return all_records


def _write_tiered(
    record: dict, qr: dict,
    quality_log, gold_out, silver_out,
):
    """FIX-C: Write record to appropriate tier file based on quality score."""
    tier = qr.get("tier", "bronze")
    if tier == "gold" and gold_out:
        gold_out.write(json.dumps(record, ensure_ascii=False) + "\n")
        gold_out.flush()
    elif tier == "silver" and silver_out:
        silver_out.write(json.dumps(record, ensure_ascii=False) + "\n")
        silver_out.flush()
    # bronze stays in main file only


# ─── DATASET BUILDER ──────────────────────────────────────────────────────────

def build_dataset(
    tickers: list[dict],
    output_file: str  = OUTPUT_FILE,
    forms: tuple      = ("10-K", "10-Q", "8-K"),
    max_per_form: int = 8,
    resume: bool      = True,
) -> int:
    llm  = get_llm()
    sess = EDGARSession()

    done_accessions: set[str] = set()
    if resume and os.path.exists(output_file):
        with open(output_file) as f:
            for line in f:
                try:
                    accno = json.loads(line).get("meta", {}).get("accession_no", "")
                    if accno:
                        done_accessions.add(accno)
                except Exception:
                    pass
        if done_accessions:
            logger.info(f"Resuming — {len(done_accessions)} accessions done")

    logger.info(f"Processing {len(tickers)} companies | forms: {forms}")
    written = 0
    pending = 0

    with open(output_file,      "a", encoding="utf-8") as out, \
         open(PENDING_FILE,     "a", encoding="utf-8") as pf,  \
         open(QUALITY_LOG_FILE, "a", encoding="utf-8") as ql,  \
         open(SANITY_LOG_FILE,  "a", encoding="utf-8") as sl,  \
         open(GOLD_FILE,        "a", encoding="utf-8") as gf,  \
         open(SILVER_FILE,      "a", encoding="utf-8") as sf:

        for company in tqdm(tickers, desc="Companies"):
            try:
                records = process_company(
                    company         = company,
                    sess            = sess,
                    llm             = llm,
                    forms           = forms,
                    max_per_form    = max_per_form,
                    done_accessions = done_accessions,
                    quality_log     = ql,
                    sanity_log      = sl,
                    gold_out        = gf,
                    silver_out      = sf,
                )
                for rec in records:
                    if rec.pop("_needs_llm", False):
                        pf.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        pf.flush()
                        pending += 1
                    else:
                        out.write(json.dumps(rec, ensure_ascii=False) + "\n")
                        out.flush()
                        written += 1
                logger.info(f"  ✓ {company['ticker']}: {len(records)} records | total: {written}")
            except Exception as e:
                logger.exception(f"  ✗ {company['ticker']} crashed: {e}")
                continue

    # Retry pending
    if pending > 0 and os.path.exists(PENDING_FILE):
        logger.info(f"\n{pending} pending — retrying...")
        retried = 0
        llm     = get_llm()
        with open(PENDING_FILE) as pf_in, \
             open(output_file, "a") as out_r, \
             open(QUALITY_LOG_FILE, "a") as ql_r, \
             open(GOLD_FILE, "a") as gf_r, \
             open(SILVER_FILE, "a") as sf_r:
            for line in pf_in:
                try:
                    rec      = json.loads(line)
                    analysis = generate_analysis(build_single_prompt(rec), llm)
                    if analysis:
                        m   = rec.get("metrics", {})
                        c   = rec.get("changes", {})
                        ht  = any(v for v in rec.get("text_sections", {}).values()
                                  if v and len(v) >= MIN_SECTION_CHARS)
                        qr  = QualityValidator.validate(analysis, m, c, ht)
                        ql_r.write(json.dumps({"retry": True, **qr}, ensure_ascii=False) + "\n")
                        rec["instruction_pair"] = build_single_pair(rec, analysis, quality_result=qr)
                        _write_tiered(rec, qr, None, gf_r, sf_r)
                        retried += 1
                    out_r.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    out_r.flush()
                    written += 1
                except Exception as e:
                    logger.warning(f"  Retry failed: {e}")
        os.remove(PENDING_FILE)
        logger.success(f"Retry: {retried}/{pending} completed")

    logger.success(f"Done. {written} records → {output_file}")
    logger.success(f"Gold tier → {GOLD_FILE} | Silver tier → {SILVER_FILE}")
    logger.success(f"Rate limiter: {_rate_limiter.summary()}")
    return written


# ══════════════════════════════════════════════════════════════════════════════
# DATASET STATS REPORTER — v15
# ══════════════════════════════════════════════════════════════════════════════

def report_dataset_stats(
    output_file:  str = OUTPUT_FILE,
    quality_file: str = QUALITY_LOG_FILE,
    sanity_file:  str = SANITY_LOG_FILE,
    gold_file:    str = GOLD_FILE,
    silver_file:  str = SILVER_FILE,
):
    if not os.path.exists(output_file):
        print("No dataset file found.")
        return

    records: list[dict] = []
    with open(output_file, encoding="utf-8") as f:
        for line in f:
            try:
                records.append(json.loads(line))
            except Exception:
                pass

    total = len(records)
    if total == 0:
        print("Dataset file is empty.")
        return

    div = "=" * 70
    print(f"\n{div}")
    print(f"DATASET STATS — v15  ({total} total records)")
    print(f"{div}")

    # Record types
    types: dict[str, int] = defaultdict(int)
    for r in records:
        rt = (r.get("meta", {}).get("record_type")
              or r.get("instruction_pair", {}).get("record_type") or "unknown")
        types[rt] += 1
    print("\nRecord types:")
    for k, v in sorted(types.items(), key=lambda x: -x[1]):
        bar = "█" * int(v / total * 40)
        print(f"  {k:<35} {v:>6}  {v/total*100:5.1f}%  {bar}")

    # FIX-A: Text extraction success
    text_filled = sum(
        1 for r in records
        if r.get("instruction_pair", {}).get("metadata", {}).get("text_sections_filled", 0) > 0
    )
    mda_filled = sum(
        1 for r in records
        if r.get("instruction_pair", {}).get("metadata", {}).get("has_mda", False)
    )
    print(f"\n{'─'*70}")
    print("FIX-A  Narrative Extraction:")
    print(f"  Records with any text section:  {text_filled:>6} / {total}  ({text_filled/total*100:.1f}%)")
    print(f"  Records with MD&A extracted:    {mda_filled:>6} / {total}  ({mda_filled/total*100:.1f}%)")
    if mda_filled == 0:
        print("  ⚠ WARNING: 0% MD&A extraction — check scraping strategies")
    elif mda_filled / total < 0.3:
        print("  ⚠ LOW: MD&A coverage below 30%")
    else:
        print("  ✓ MD&A extraction functioning")

    # FIX-B: Rate limit stats
    print(f"\n{'─'*70}")
    print("FIX-B  Rate Limit Manager:")
    print(f"  Total 429s encountered:  {_rate_limiter.total_429s:>6}")
    print(f"  Records skipped:         {_rate_limiter.total_skipped:>6}")
    print(f"  Circuit breaker trips:   {'YES — check logs' if _rate_limiter.total_429s >= LLM_CIRCUIT_BREAK_N else 'NO'}")

    # FIX-C: Gold/Silver tier counts
    gold_count   = 0
    silver_count = 0
    if os.path.exists(gold_file):
        with open(gold_file) as f:
            gold_count = sum(1 for line in f if line.strip())
    if os.path.exists(silver_file):
        with open(silver_file) as f:
            silver_count = sum(1 for line in f if line.strip())

    tier_counts: dict[str, int] = defaultdict(int)
    for r in records:
        tier = r.get("instruction_pair", {}).get("metadata", {}).get("quality_tier", "unknown")
        tier_counts[tier] += 1

    print(f"\n{'─'*70}")
    print("FIX-C  Quality Tier Distribution (Gold Filter):")
    print(f"  🥇 Gold   (score ≥{GOLD_QUALITY_THRESHOLD}):   {gold_count:>6} records in {GOLD_FILE}")
    print(f"  🥈 Silver (score ≥{SILVER_QUALITY_THRESHOLD}): {silver_count:>6} records in {SILVER_FILE}")
    for tier in ("gold", "silver", "bronze", "unknown"):
        v = tier_counts.get(tier, 0)
        if v:
            icon = {"gold": "🥇", "silver": "🥈", "bronze": "🥉"}.get(tier, "?")
            print(f"  {icon} {tier:<10} {v:>6}  ({v/total*100:.1f}%)")

    # FIX-D: Abnormal return coverage
    ar_records = sum(
        1 for r in records
        if r.get("meta", {}).get("abnormal_return") is not None
    )
    ar_reactions: dict[str, int] = defaultdict(int)
    ar_values = []
    for r in records:
        ar = r.get("meta", {}).get("abnormal_return", {})
        if ar:
            ar_reactions[ar.get("market_reaction", "unknown")] += 1
            v = ar.get("abnormal_return_pct")
            if v is not None:
                ar_values.append(v)

    print(f"\n{'─'*70}")
    print("FIX-D  Abnormal Return Coverage (Yahoo Finance):")
    print(f"  Records with abnormal return:  {ar_records:>6} / {total}  ({ar_records/total*100:.1f}%)")
    if ar_values:
        avg_ar = sum(ar_values) / len(ar_values)
        print(f"  Avg abnormal return:           {avg_ar:+.2f}%")
        print(f"  Market reaction distribution:")
        for reaction in ["strong_positive", "positive", "neutral", "negative", "strong_negative"]:
            cnt = ar_reactions.get(reaction, 0)
            if cnt:
                print(f"    {reaction:<20} {cnt:>5}")

    # FIX-1: De-cumulation
    decumed    = sum(1 for r in records if r.get("meta", {}).get("decumulation_applied") is True)
    q2q3_recs  = sum(1 for r in records
                     if r.get("meta", {}).get("fiscal_quarter") in (2, 3)
                     and "10-Q" in str(r.get("meta", {}).get("form_type", "")))
    print(f"\n{'─'*70}")
    print("FIX-1  De-cumulation:")
    print(f"  Applied:        {decumed:>6} / {total}  ({decumed/total*100:.1f}%)")
    print(f"  Eligible Q2+Q3: {q2q3_recs:>6}")
    if q2q3_recs > 0:
        cov    = decumed / q2q3_recs * 100
        status = "✓ COMPLETE" if cov >= 90 else "⚠ PARTIAL" if cov >= 50 else "✗ LOW"
        print(f"  Coverage:       {cov:.1f}%  {status}")

    # FIX-2: Quarter-aligned YoY
    q_aligned = sum(1 for r in records if r.get("meta", {}).get("quarter_aligned_yoy") is True)
    print(f"\n{'─'*70}")
    print("FIX-2  Quarter-Aligned YoY:")
    print(f"  Applied: {q_aligned:>6} / {total}  ({q_aligned/total*100:.1f}%)")

    # FIX-3: Sanity
    sanity_counts: dict[str, int] = defaultdict(int)
    for r in records:
        sanity_counts[r.get("meta", {}).get("sanity_action", "unknown")] += 1
    sanity_log_entries = []
    if os.path.exists(sanity_file):
        with open(sanity_file, encoding="utf-8") as sf_in:
            for line in sf_in:
                try:
                    sanity_log_entries.append(json.loads(line))
                except Exception:
                    pass
    print(f"\n{'─'*70}")
    print("FIX-3  Sanity Actions:")
    for action in ["pass", "flag", "recalculate", "block", "unknown"]:
        cnt = sanity_counts.get(action, 0)
        if cnt:
            icon = {"pass": "✓", "flag": "⚠", "recalculate": "↻", "block": "✗"}.get(action, "?")
            print(f"  {icon} {action:<15} {cnt:>6}  ({cnt/total*100:.1f}%)")
    if sanity_log_entries:
        blocked_flags: dict[str, int] = defaultdict(int)
        for entry in sanity_log_entries:
            for flag in entry.get("flags", []):
                blocked_flags[flag.split(":")[0]] += 1
        if blocked_flags:
            print("  Flag types:")
            for rule, cnt in sorted(blocked_flags.items(), key=lambda x: -x[1]):
                print(f"    {rule:<40} {cnt:>5}")

    # FIX-4: Period type
    pt_counts: dict[str, int] = defaultdict(int)
    for r in records:
        pt_counts[r.get("meta", {}).get("period_type", "unknown")] += 1
    print(f"\n{'─'*70}")
    print("FIX-4  Period Type Distribution:")
    for k, v in sorted(pt_counts.items(), key=lambda x: -x[1]):
        print(f"  {k:<25} {v:>6}  ({v/total*100:.1f}%)")

    # Edge cases
    edge_counts: dict[str, int] = defaultdict(int)
    for r in records:
        for ec in (r.get("instruction_pair", {}).get("metadata", {}).get("edge_cases", [])
                   or r.get("signals", {}).get("edge_cases_detected", [])):
            edge_counts[ec] += 1
    if edge_counts:
        print(f"\n{'─'*70}")
        print("Edge Case Coverage:")
        for k, v in sorted(edge_counts.items(), key=lambda x: -x[1]):
            print(f"  {k:<40} {v:>5}")

    # Quality scores
    qscores:      list[float] = []
    hall_flags:   int         = 0
    delta_issues: int         = 0
    style_fails:  int         = 0

    if os.path.exists(quality_file):
        with open(quality_file, encoding="utf-8") as qf:
            for line in qf:
                try:
                    q  = json.loads(line)
                    qs = q.get("quality_score")
                    if qs is not None:
                        qscores.append(float(qs))
                    if q.get("hallucination", {}).get("flag"):
                        hall_flags += 1
                    if not q.get("delta_consistency", {}).get("clean", True):
                        delta_issues += 1
                    st = q.get("style", {})
                    if not st.get("density_ok", True) or not st.get("length_ok", True):
                        style_fails += 1
                except Exception:
                    pass

    print(f"\n{'─'*70}")
    print("Quality Validation:")
    if qscores:
        n       = len(qscores)
        avg     = sum(qscores) / n
        below75 = sum(1 for q in qscores if q < 75)
        above90 = sum(1 for q in qscores if q >= 90)
        median  = sorted(qscores)[n // 2]
        print(f"  Validated:              {n:>6}")
        print(f"  Avg score:              {avg:>6.1f} / 100")
        print(f"  Median score:           {median:>6.1f} / 100")
        print(f"  Gold (≥{GOLD_QUALITY_THRESHOLD}):           {above90:>6}  ({above90/n*100:.1f}%)")
        print(f"  Below silver (<{SILVER_QUALITY_THRESHOLD}):   {below75:>6}  ({below75/n*100:.1f}%)")
        print(f"  Hallucination flags:    {hall_flags:>6}  ({hall_flags/n*100:.1f}%)")
        print(f"  Delta inconsistencies:  {delta_issues:>6}  ({delta_issues/n*100:.1f}%)")
        print(f"  Style failures:         {style_fails:>6}  ({style_fails/n*100:.1f}%)")
        buckets = {"90-100": 0, "80-89": 0, "70-79": 0, "60-69": 0, "<60": 0}
        for qs in qscores:
            if   qs >= 90: buckets["90-100"] += 1
            elif qs >= 80: buckets["80-89"]  += 1
            elif qs >= 70: buckets["70-79"]  += 1
            elif qs >= 60: buckets["60-69"]  += 1
            else:          buckets["<60"]    += 1
        print("\n  Score buckets:")
        for bucket, cnt in buckets.items():
            bar = "█" * int(cnt / max(n, 1) * 30)
            print(f"    {bucket:<8}  {cnt:>5}  {bar}")
    else:
        print("  No quality log data.")

    # Reasoning types
    rtype_counts: dict[str, int] = defaultdict(int)
    for r in records:
        for rt in r.get("instruction_pair", {}).get("metadata", {}).get("reasoning_types", []):
            rtype_counts[rt] += 1
    if rtype_counts:
        print(f"\n{'─'*70}")
        print("Reasoning Type Coverage:")
        for k, v in sorted(rtype_counts.items(), key=lambda x: -x[1]):
            bar = "█" * int(v / total * 30)
            print(f"  {k:<30} {v:>6}  {v/total*100:5.1f}%  {bar}")

    # Pipeline health
    print(f"\n{'─'*70}")
    print("PIPELINE HEALTH:")
    checks = [
        ("FIX-A MD&A extraction > 0%",             mda_filled > 0),
        ("FIX-B 429 skip rate < 10%",              (_rate_limiter.total_skipped / max(total, 1)) < 0.10),
        ("FIX-C Gold tier records exist",          gold_count > 0),
        ("FIX-D Abnormal return coverage > 50%",   ar_records / max(total, 1) > 0.5),
        ("FIX-1 De-cumulation applied",            decumed > 0 or q2q3_recs == 0),
        ("FIX-2 Quarter-aligned YoY in use",       q_aligned > 0),
        ("FIX-3 Sanity blocking active",           "block" in sanity_counts or "flag" in sanity_counts),
        ("FIX-4 period_type < 10% unknown",        pt_counts.get("unknown", 0) / max(total, 1) < 0.1),
        ("Quality avg ≥ 75",                       (sum(qscores) / len(qscores) >= 75) if qscores else False),
        ("Hallucination rate < 10%",               (hall_flags / len(qscores) < 0.10) if qscores else True),
    ]
    for label, ok in checks:
        print(f"  {'✓' if ok else '✗'}  {label}")

    print(f"\n{div}\n")



In [4]:

# ══════════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    tickers = get_nasdaq100_tickers()
    tickers = tickers[:3]   # remove [:3] for full run

    print("=" * 70)
    print("SEC Fine-Tuning Dataset Builder  v15  (RESEARCH-GRADE)")
    print("=" * 70)
    print(f"  Companies     : {len(tickers)}")
    print(f"  Forms         : 10-K + 10-Q + 8-K")
    print(f"  Output        : {OUTPUT_FILE}")
    print(f"  Gold tier     : {GOLD_FILE}")
    print(f"  Silver tier   : {SILVER_FILE}")
    print(f"  Quality log   : {QUALITY_LOG_FILE}")
    print(f"  Sanity log    : {SANITY_LOG_FILE}")


    count = build_dataset(
        tickers      = tickers,
        output_file  = OUTPUT_FILE,
        forms        = ("10-K", "10-Q", "8-K"),
        max_per_form = 8,
        resume       = True,
    )

    report_dataset_stats(OUTPUT_FILE, QUALITY_LOG_FILE, SANITY_LOG_FILE, GOLD_FILE, SILVER_FILE)
    print(f"\nDone: {count} records → {OUTPUT_FILE}")


/var/folders/fx/61s94vks26j1dz_f_1bkfbhw0000gn/T/ipykernel_98517/942191937.py:20: DeprecationWarning: The 'max_tokens' parameter is deprecated and will be removed in a future version. Please use 'max_completion_tokens' instead.
  return ChatNVIDIA(
00:25:18 | INFO    | Processing 3 companies | forms: ('10-K', '10-Q', '8-K')


SEC Fine-Tuning Dataset Builder  v15  (RESEARCH-GRADE)
  Companies     : 3
  Forms         : 10-K + 10-Q + 8-K
  Output        : sec_finetune_v15.jsonl
  Gold tier     : sec_finetune_v15_gold.jsonl
  Silver tier   : sec_finetune_v15_silver.jsonl
  Quality log   : sec_quality_v15.jsonl
  Sanity log    : sec_sanity_v15.jsonl


Companies:   0%|          | 0/3 [00:00<?, ?it/s]00:25:18 | INFO    | ══ AAPL (Apple Inc) ══
00:25:18 | INFO    | CIK table: 10,380 entries
00:25:20 | INFO    |   XBRL: 503 GAAP + 2 DEI
00:25:21 | INFO    |   10-K: 5 filings
00:25:21 | INFO    |   ▶ 10-K | 2021-09-25 | 0000320193-21-000105
00:25:26 | INFO    |     Strategy-1 sections: 1/6 (item_score=30)
00:25:26 | INFO    |     Strategy-2 Risk fallback: 3831 chars
00:25:26 | INFO    |     Final sections: 2/6 | MD&A quality: poor (85 chars)
00:25:26 | INFO    |     Metrics: 47 | decum: False
00:25:35 | INFO    |     ✓ [FULL|Q=100|tier=gold|decum=False|qa=False]: 1354 chars
00:25:35 | INFO    |   ▶ 10-K | 2022-09-24 | 0000320193-22-000108
00:25:38 | INFO    |     Strategy-1 sections: 1/6 (item_score=29)
00:25:38 | INFO    |     Strategy-2 Risk fallback: 3897 chars
00:25:38 | INFO    |     Final sections: 2/6 | MD&A quality: poor (85 chars)
00:25:38 | INFO    |     Metrics: 47 | decum: False
00:25:48 | INFO    |     ✓ [FULL|Q=75|tier=silv


DATASET STATS — v15  (104 total records)

Record types:
  single_period                           48   46.2%  ██████████████████
  multi_year_3yr                          31   29.8%  ███████████
  unknown                                 15   14.4%  █████
  multi_year                               6    5.8%  ██
  negative_reasoning                       4    3.8%  █

──────────────────────────────────────────────────────────────────────
FIX-A  Narrative Extraction:
  Records with any text section:      48 / 104  (46.2%)
  Records with MD&A extracted:        48 / 104  (46.2%)
  ✓ MD&A extraction functioning

──────────────────────────────────────────────────────────────────────
FIX-B  Rate Limit Manager:
  Total 429s encountered:       0
  Records skipped:              0
  Circuit breaker trips:   NO

──────────────────────────────────────────────────────────────────────
FIX-C  Quality Tier Distribution (Gold Filter):
  🥇 Gold   (score ≥90):       25 records in sec_finetune_v15_gold.jso